In [ ]:


Looking at your code, I can identify several issues that prevent maximum part planning and cause machines to sit idle or underused. Here are the key problems and fixes:

## Key Issues Found

1. **Parts whose planned qty is already met on one machine get re-planned on other machines** (violating your constraint) — the `already_planned` set isn't checked properly in the strategic buffer filler
2. **OPD cap is too restrictive** — parts at ceiling get skipped even when machines are idle
3. **The enforcer doesn't aggressively seek NEW unplanned parts** — it extends existing parts first, leaving machines underused
4. **Strangers/Repeaters are over-restricted** — single-machine rule prevents them from filling idle machines even when they haven't been planned at all
5. **Strategic buffer filler re-plans parts already at planned qty on new machines** instead of finding fresh parts

Here's the corrected and improved version with critical fixes:

```python
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date

# =============================================================
#  Smart APS V11  —  Maximum Parts, Zero Idle Release
#
#  Changes vs V10:
#  ─────────────────────────────────────────────────────────────
#  FIX-17 : Strategic buffer filler now tracks "planned_qty_met"
#            per part. Parts whose daily indent is already fully
#            covered are NEVER assigned to a NEW machine —
#            only extended on their EXISTING machine(s).
#  FIX-18 : Enforcer S2 now aggressively seeks ALL unplanned
#            parts before extending existing ones. Priority:
#            1. Unplanned parts with zero inventory
#            2. Unplanned parts below safety
#            3. Unplanned parts below target
#            4. Then extend existing parts
#  FIX-19 : "Planned qty met" tracking — a part whose total
#            production across all machines >= daily_indent is
#            marked. It can still be EXTENDED on its current
#            machine(s) but NEVER assigned to a new machine.
#  FIX-20 : Machine idle elimination — after all passes, any
#            machine with > 0.5h idle gets a forced extension
#            of its highest-priority existing part up to
#            STRATEGIC_BUFFER_DAYS. No new CO, no new part.
#  FIX-21 : Parts not yet planned anywhere get absolute priority
#            in the enforcer over parts already running.
#  FIX-22 : Better candidate sorting in enforcer — parts with
#            ZERO planned qty today always rank first.
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS
# =============================================================

PLANNING_DATE = date(2026, 4, 9)
INDENT_MONTH  = date(2026, 4, 1)

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS          = 22
AVAILABLE_HOURS_EXTENDED = 23
MIN_RUN_HOURS            = 4
MACHINE_STATE_FILE       = "machine_state.json"

MIN_DAILY_INDENT         = 150
MIN_INDENT_HOURS         = 4.0

SAFETY_DAYS              = 3
TARGET_DAYS              = 5

OPD_SCENARIO_0 = 3.0
OPD_SCENARIO_1 = 3.0
OPD_SCENARIO_2 = 4.0
OPD_SCENARIO_3 = 5.0

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

UTIL_TARGET_PCT  = 90.0
COLOR_PURGE_HRS  = 10 / 60.0

RUNNER_PRIORITY_DAYS = 2.0

MAX_DAILY_CO             = 25

FORWARD_LOOK_DAYS        = 7

TERMINAL_THRESHOLD = {
    "Runner":   1.0,
    "Repeater": 0.5,
    "Stranger": 0.25,
}

STRATEGIC_BUFFER_DAYS        = 7
STRATEGIC_PRIORITY_DISCOUNT  = 0.5
EFFICIENCY_MODE_UTIL_FLOOR   = 95.0

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
terminal_path   = "C:/Users/Ex0164/Important codes/terminals and raw marterial - vt.xlsx"
output_path     = f"Smart_APS_V11_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

# =============================================================
# SECTION 4 — WORKING DAYS
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V11  —  Maximum Parts, Zero Idle Release")
print(f"  Planning date  : {PLANNING_DATE}")
print(f"  Indent month   : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days   : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"  Safety floor   : {SAFETY_DAYS} days  |  Target ceiling : {TARGET_DAYS} days")
print(f"  Runner priority: inv < {RUNNER_PRIORITY_DAYS}×daily")
print(f"  Max daily CO   : {MAX_DAILY_CO}")
print(f"  Forward look   : {FORWARD_LOOK_DAYS} days (advisory)")
print(f"{'='*65}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
vt_parts_raw         = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix            = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw            = pd.read_excel(changeover_path, sheet_name="VT_Changeover")
vt_machine_count_raw = pd.read_excel(matrix_path,     sheet_name="VT_Machine_Part_Count")

try:
    vt_fixed_raw = pd.read_excel(matrix_path, sheet_name="VT_Fixed")
    print(f"  VT_Fixed sheet loaded  ({len(vt_fixed_raw)} rows)")
except Exception as _fe:
    vt_fixed_raw = None
    print(f"  WARNING: VT_Fixed sheet not found ({_fe})")

try:
    vt_terminals_raw      = pd.read_excel(terminal_path, sheet_name="VT_Terminals")
    vt_terminal_avail_raw = pd.read_excel(terminal_path, sheet_name="VT_Terminal_Inventory")
    print(f"  Terminal data loaded from  : {terminal_path}")
except FileNotFoundError:
    vt_terminals_raw      = None
    vt_terminal_avail_raw = None
    print(f"  WARNING: terminal_path not found — terminal constraint DISABLED.")
except Exception as _te:
    vt_terminals_raw      = None
    vt_terminal_avail_raw = None
    print(f"  WARNING: Could not load terminal data ({_te}) — constraint DISABLED.")

# =============================================================
# SECTION 6 — PARSE VT SHEET
# =============================================================

def find_col(df, name, sheet):
    match = next(
        (c for c in df.columns if str(c).strip().lower() == name.lower()), None
    )
    if match is None:
        raise ValueError(
            f"Column '{name}' not found in sheet '{sheet}'.\n"
            f"Available: {list(df.columns)}"
        )
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")
vt_col_tools     = find_col(vt_parts_raw, "Tools",      "VT")
vt_col_color     = find_col(vt_parts_raw, "Color",      "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()

data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]

data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()

print(f"  VT parts in sheet       : {len(data)}")
print(f"  Parts with valid rate   : {len(data_valid)}")

# =============================================================
# SECTION 7 — LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)

tools_available = {}
part_color      = {}

for _, row in data.iterrows():
    p = str(row["Material"]).strip()
    v = row[vt_col_tools]
    tools_available[p] = (
        max(1, int(float(v)))
        if pd.notna(v) and str(v).strip() != ""
        else 1
    )
    c = row[vt_col_color]
    part_color[p] = (
        str(c).strip().upper()
        if pd.notna(c) and str(c).strip() not in ("", "nan")
        else "UNKNOWN"
    )

ALL_KNOWN_COLORS = dict(part_color)

color_groups = {}
for p, c in part_color.items():
    color_groups.setdefault(c, []).append(p)
print(f"  Distinct colours        : {len(color_groups)}")
for col, pts in sorted(color_groups.items()):
    print(f"    {col:<20} → {len(pts)} part(s)")

indent_daily = {
    p: round(qty / WORKING_DAYS, 4)
    for p, qty in indent_monthly.items()
}

today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}

# =============================================================
# SECTION 7A — FIXED MACHINE CONSTRAINT
# =============================================================

def build_fixed_machine_dicts(df):
    pfm = {}
    mfp = {}
    if df is None or df.empty:
        return pfm, mfp
    machine_col = next(
        (c for c in df.columns if str(c).strip().lower() == "machine"), None
    )
    if machine_col is None:
        print("  WARNING: VT_Fixed has no 'Machine' column — fixed constraint disabled")
        return pfm, mfp
    part_cols = [c for c in df.columns if str(c).strip().lower() != "machine"]
    if not part_cols:
        print("  WARNING: VT_Fixed has no part columns — fixed constraint disabled")
        return pfm, mfp
    for _, row in df.iterrows():
        machine = row[machine_col]
        if pd.isna(machine) or str(machine).strip() == "":
            continue
        m = str(machine).strip()
        for col in part_cols:
            val = row[col]
            if pd.isna(val) or str(val).strip() in ("", "nan"):
                continue
            p = str(val).strip()
            if p in pfm:
                print(f"  WARNING: Part '{p}' in VT_Fixed more than once — keeping {pfm[p]}")
                continue
            pfm[p] = m
            mfp.setdefault(m, []).append(p)
    return pfm, mfp


part_fixed_machine, machine_fixed_parts = build_fixed_machine_dicts(vt_fixed_raw)

print(f"  Fixed machine mappings  : {len(part_fixed_machine)} parts")
if part_fixed_machine:
    for m, parts in sorted(machine_fixed_parts.items()):
        print(f"    {m:<25} ← {', '.join(parts)}")

# =============================================================
# SECTION 7A2 — FIXED MACHINE PHASE HELPERS
# =============================================================

def fixed_machine_phase(machine, current_inventory):
    fixed_parts = machine_fixed_parts.get(machine, [])
    if not fixed_parts:
        return "B"
    for p in fixed_parts:
        daily = indent_daily.get(p, 0)
        inv   = current_inventory.get(p, 0)
        if daily > 0 and inv < SAFETY_DAYS * daily:
            return "A"
    return "B"


def pick_fixed_part_for_today(machine, current_inventory, machine_last_part):
    fixed_parts = machine_fixed_parts.get(machine, [])
    candidates  = []
    for p in fixed_parts:
        daily = indent_daily.get(p, 0)
        r_val = rate.get(p, 1)
        inv   = current_inventory.get(p, 0)
        if daily <= 0:
            continue
        days_cov     = inv / daily
        hours_needed = daily / r_val if r_val > 0 else 0
        candidates.append((p, days_cov, hours_needed))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (round(x[1], 4), -x[2]))
    if len(candidates) >= 2:
        top, second = candidates[0], candidates[1]
        if abs(top[1] - second[1]) < 0.01:
            last_ran = machine_last_part.get(machine)
            if last_ran == top[0]:
                return second[0]
    return candidates[0][0]


def fixed_part_run_hours(part, phase):
    daily = indent_daily.get(part, 0)
    r_val = rate.get(part, 1)
    if daily <= 0 or r_val <= 0:
        return AVAILABLE_HOURS
    indent_hrs = daily / r_val
    if phase == "A":
        return AVAILABLE_HOURS_EXTENDED if indent_hrs > 20.0 else AVAILABLE_HOURS
    else:
        return max(MIN_RUN_HOURS, indent_hrs)

# =============================================================
# SECTION 7B — TERMINAL CONSTRAINT
# =============================================================

def _build_part_terminals(df):
    result = {}
    if df is None or df.empty:
        return result
    part_col = next(
        (c for c in df.columns if str(c).strip().lower() in ("part", "material")), None
    )
    if part_col is None:
        print("  WARNING: VT_Terminals has no 'Part'/'Material' column")
        return result
    terminal_cols = [
        c for c in df.columns
        if str(c).strip().lower() not in ("part", "material")
    ]
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        p = str(part).strip()
        terminals = []
        for col in terminal_cols:
            val = row[col]
            if pd.notna(val) and str(val).strip() not in ("", "nan"):
                terminals.append(str(val).strip().upper())
        if terminals:
            result[p] = terminals
    return result


def _build_terminal_status(df):
    result = {}
    if df is None or df.empty:
        return result
    term_col = next(
        (c for c in df.columns if str(c).strip().lower() == "terminal"), None
    )
    inv_col = next(
        (c for c in df.columns if str(c).strip().lower() == "inventory"), None
    )
    if term_col is None or inv_col is None:
        print("  WARNING: VT_Terminal_Inventory missing 'Terminal' or 'Inventory' column")
        return result
    for _, row in df.iterrows():
        t = row[term_col]
        v = row[inv_col]
        if pd.isna(t) or str(t).strip() == "":
            continue
        key = str(t).strip().upper()
        try:
            qty = float(v) if pd.notna(v) else 0.0
        except (ValueError, TypeError):
            qty = 0.0
        result[key] = qty
    return result


part_terminals  = _build_part_terminals(vt_terminals_raw)
terminal_status = _build_terminal_status(vt_terminal_avail_raw)

if terminal_status:
    zero_count  = sum(1 for v in terminal_status.values() if v <= 0)
    avail_count = len(terminal_status) - zero_count
    print(f"  Terminals loaded : {len(terminal_status)} total  |  "
          f"{avail_count} with stock  |  {zero_count} at ZERO inventory")
else:
    print(f"  Terminals        : no data loaded — constraint inactive")


def terminal_blocked(part, category_override=None):
    required = part_terminals.get(part, [])
    if not required:
        return False, ""
    cat   = category_override or part_category.get(part, "Stranger")
    daily = indent_daily.get(part, 0.0)
    mult  = TERMINAL_THRESHOLD.get(cat, 0.25)
    threshold = mult * daily
    blocking = []
    for t in required:
        t_inv = terminal_status.get(t, 0)
        if t_inv < threshold:
            blocking.append(
                f"{t}(inv={t_inv:.0f} < need={threshold:.0f} [{cat} ×{mult}])"
            )
    if blocking:
        return True, (
            f"Terminal inadequate: {', '.join(blocking)}  "
            f"(requires: {', '.join(required)})"
        )
    return False, ""

# =============================================================
# SECTION 7C — SKIP RULES
# =============================================================

def should_skip(part):
    daily   = indent_daily.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    if daily <= MIN_DAILY_INDENT:
        return True, f"Daily indent {daily:.2f} ≤ {MIN_DAILY_INDENT} threshold"

    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, f"Whole monthly indent = {indent_hrs:.2f}h ≤ {MIN_INDENT_HOURS}h threshold"

    inv = inventory.get(part, 0.0)
    if daily > 0 and inv >= TARGET_DAYS * daily:
        return True, (
            f"Inventory ({inv:.0f}) ≥ {TARGET_DAYS}-day target "
            f"({TARGET_DAYS * daily:.0f} pcs) — at ceiling, skip today"
        )

    t_blocked, t_reason = terminal_blocked(part)
    if t_blocked:
        return True, t_reason

    return False, ""


def is_hard_skip(part):
    daily   = indent_daily.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    if daily <= MIN_DAILY_INDENT:
        return True
    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True
    if terminal_blocked(part)[0]:
        return True
    return False

# =============================================================
# SECTION 7D — CHANGEOVER TIMES
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover          = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

# =============================================================
# SECTION 7E — MACHINE PART COUNT
# =============================================================

def build_machine_part_count(df):
    mpc = {}
    machine_col = next(
        (c for c in df.columns if str(c).strip().lower() == "machine"), None
    )
    count_col = next(
        (c for c in df.columns if str(c).strip().lower() == "part_count"), None
    )
    if machine_col is None or count_col is None:
        print("  WARNING: VT_Machine_Part_Count missing columns")
        return {}
    for _, row in df.iterrows():
        m = str(row[machine_col]).strip()
        v = row[count_col]
        if m and pd.notna(v):
            try:
                mpc[m] = int(float(v))
            except (ValueError, TypeError):
                pass
    return mpc

machine_part_count = build_machine_part_count(vt_machine_count_raw)
max_part_count     = max(machine_part_count.values(), default=1) or 1
print(f"  Machine part counts loaded: {len(machine_part_count)} machines")

# =============================================================
# SECTION 7F — PART CATEGORY
# =============================================================

def build_category(df):
    cat      = {}
    part_col = next(
        (c for c in df.columns if str(c).strip().lower() == "part"), None
    )
    cat_col  = next(
        (c for c in df.columns if str(c).strip().lower() == "category"), None
    )
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category  = build_category(vt_parts_raw)
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            with open(MACHINE_STATE_FILE) as f:
                content = f.read().strip()
            if not content:
                print(f"  Machine state : file empty — first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                print(f"  Machine state : corrupt — first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            unknown = [p for p in state.values() if p not in part_color]
            if unknown:
                print(f"  Machine state : {len(unknown)} part(s) not in today's sheet:")
                for p in unknown:
                    print(f"    {p} — kept for CO calculation (color unknown → purge assumed)")
                    ALL_KNOWN_COLORS[p] = "NEEDS_PURGE"
            print(f"  Machine state loaded  ({len(state)} machines)")
            return state
        except Exception as e:
            print(f"  Machine state : error ({e}) — first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip or daily == 0:
            continue
        coverage.append(inv / daily)

    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < SAFETY_DAYS)

    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} active parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical  |  {n-critical} have buffer"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {SAFETY_DAYS}-day safety floor"
    else:
        return 3, f"SCENARIO 3 — All {n} parts healthy (≥{SAFETY_DAYS} days safety floor)"

# =============================================================
# SECTION 11 — OPD CAP
# =============================================================

def opd_cap(scenario_id):
    scenario_opd = {
        0: OPD_SCENARIO_0,
        1: OPD_SCENARIO_1,
        2: OPD_SCENARIO_2,
        3: OPD_SCENARIO_3,
    }.get(scenario_id, OPD_SCENARIO_2)
    return min(scenario_opd, TARGET_DAYS)

# =============================================================
# SECTION 12 — PRIORITY SCORING
# =============================================================

def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap_score = min(1.0, max(0.0, (TARGET_DAYS - days_cov) / TARGET_DAYS))
        velocity_raw = daily / max(float(inv), 1.0) if daily > 0 else 0.0
        rows.append({
            "part":         p,
            "inv":          inv,
            "daily":        daily,
            "days_cov":     days_cov,
            "cat":          cat,
            "gap_score":    gap_score,
            "velocity_raw": velocity_raw,
        })

    if not rows:
        return {}, []

    max_daily    = max(r["daily"]        for r in rows) or 1.0
    max_velocity = max(r["velocity_raw"] for r in rows) or 1.0
    scores, score_rows = {}, []

    for r in rows:
        p = r["part"]
        gap_pct      = r["gap_score"] * 100.0
        velocity_pct = (r["velocity_raw"] / max_velocity) * 100.0
        urgency_score  = 0.60 * gap_pct + 0.40 * velocity_pct
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100.0
        final_score = (
            W_URGENCY  * urgency_score +
            W_CATEGORY * category_score +
            W_INDENT   * indent_score
        )
        scores[p] = round(final_score, 2)
        fixed_m    = part_fixed_machine.get(p, "—")
        score_rows.append({
            "Part":              p,
            "Category":          r["cat"],
            "Color":             part_color.get(p, "UNKNOWN"),
            "Fixed_Machine":     fixed_m,
            "Tools":             tools_available.get(p, 1),
            "Inventory_Now":     round(r["inv"], 0),
            "Daily_Indent":      round(r["daily"], 2),
            "Days_Coverage":     round(r["days_cov"], 2),
            "Safety_Floor":      SAFETY_DAYS,
            "Target_Ceiling":    TARGET_DAYS,
            "Buffer_Status":     (
                "CRITICAL"     if r["days_cov"] < 1 else
                "BELOW_SAFETY" if r["days_cov"] < SAFETY_DAYS else
                "BUILDING"     if r["days_cov"] < TARGET_DAYS else
                "AT_TARGET"
            ),
            "Gap_Score_Pct":     round(gap_pct, 1),
            "Velocity_Score_Pct":round(velocity_pct, 1),
            "Urgency_Score":     round(urgency_score, 1),
            "Category_Score":    category_score,
            "Indent_Score":      round(indent_score, 1),
            "Final_Score":       round(final_score, 2),
        })

    return scores, score_rows

# =============================================================
# SECTION 13 — COLOUR-AWARE CHANGEOVER HELPER
# =============================================================

def _co_hrs_for(part, machine, machine_last_part):
    last = machine_last_part.get(machine)
    if last is None or last == part:
        return 0.0
    base_co    = vt_changeover.get(machine, DEFAULT_CHANGEOVER_HRS)
    last_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN")
    new_color  = part_color.get(part, "UNKNOWN")
    if last_color == "NEEDS_PURGE":
        return base_co + COLOR_PURGE_HRS
    purge = (
        COLOR_PURGE_HRS
        if last_color != new_color
        and last_color not in ("UNKNOWN",)
        and new_color  not in ("UNKNOWN",)
        else 0.0
    )
    return base_co + purge

# =============================================================
# SECTION 14 — MACHINE RANKER
# =============================================================

def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, inv_days,
                  exclude_fixed_machines=True):
    category  = part_category.get(part, "Stranger")
    new_color = part_color.get(part, "UNKNOWN")
    fixed_m   = part_fixed_machine.get(part)
    is_fixed  = fixed_m is not None

    if is_fixed:
        runner_lock = False
    else:
        runner_lock = (category == "Runner" and inv_days < RUNNER_PRIORITY_DAYS)

    effective_candidates = []
    for m in machines_to_try:
        if exclude_fixed_machines and m in machine_fixed_parts:
            if not is_fixed:
                continue
            elif m != fixed_m:
                continue
        effective_candidates.append(m)

    if is_fixed and fixed_m in effective_candidates:
        used_f = machine_hours.get(fixed_m, 0)
        free_f = round(AVAILABLE_HOURS - used_f, 4)
        co_f   = _co_hrs_for(part, fixed_m, machine_last_part)
        eff_f  = round(free_f - co_f, 4)

        if eff_f >= MIN_RUN_HOURS:
            fallback = _rank_normal(
                part,
                [m for m in effective_candidates if m != fixed_m],
                machine_hours, machine_last_part, new_color, runner_lock
            )
            return [(fixed_m, co_f, eff_f, -1.0)] + fallback, runner_lock

        remaining = [m for m in effective_candidates if m != fixed_m]
        return _rank_normal(
            part, remaining, machine_hours, machine_last_part,
            new_color, runner_lock
        ), runner_lock

    return _rank_normal(
        part, effective_candidates, machine_hours, machine_last_part,
        new_color, runner_lock
    ), runner_lock


def _rank_normal(part, machines_to_try, machine_hours,
                 machine_last_part, new_color, runner_lock):
    ranked = []
    for m in machines_to_try:
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue

        if last is None or last == part:
            co_hrs      = 0.0
            color_bonus = 0.0
        else:
            base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            last_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN")
            same_color = (
                last_color == new_color
                and last_color not in ("UNKNOWN", "NEEDS_PURGE")
                and new_color  not in ("UNKNOWN",)
            )
            purge      = (
                0.0 if same_color else (
                    COLOR_PURGE_HRS
                    if last_color not in ("UNKNOWN",) and new_color not in ("UNKNOWN",)
                    else 0.0
                )
            )
            co_hrs      = base_co + purge
            color_bonus = -0.08 if same_color else 0.0

        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue

        part_count      = machine_part_count.get(m, max_part_count)
        count_score     = part_count / max_part_count
        co_penalty      = (co_hrs / AVAILABLE_HOURS) * 0.3
        util_penalty    = (used  / AVAILABLE_HOURS) * 0.2
        same_part_bonus = -0.15 if (last == part) else 0.0
        cost            = (count_score + co_penalty + util_penalty
                           + same_part_bonus + color_bonus)

        ranked.append((m, co_hrs, effective_free, cost))

    ranked.sort(key=lambda x: x[3])
    return ranked

# Global set — Phase A machines
_phase_a_machines: set = set()

# =============================================================
# FIX-19: PLANNED QTY TRACKING
# =============================================================
# This dict tracks total production qty per part across all machines.
# Used to determine if a part's daily indent is already met.
# Parts with met indent should NOT be assigned to NEW machines —
# only extended on existing machines.

def _get_part_total_qty(part, plan):
    """Get total planned qty for a part across all machines."""
    return sum(float(r.get("Production_Qty", 0)) for r in plan if r["Part"] == part)

def _get_part_machines(part, plan):
    """Get set of machines a part is currently planned on."""
    return {r["Machine"] for r in plan if r["Part"] == part}

def _is_indent_met(part, plan):
    """Check if a part's daily indent is met by current planned qty."""
    daily = indent_daily.get(part, 0)
    if daily <= 0:
        return True
    total_qty = _get_part_total_qty(part, plan)
    return total_qty >= (daily - 0.5)

# =============================================================
# SECTION 14B — INTRA-MACHINE CO RESEQUENCING
# =============================================================

def resequence_machine_rows(plan, machine_last_part_yesterday):
    from collections import defaultdict
    machine_rows = defaultdict(list)
    other_rows   = []
    for row in plan:
        m = row.get("Machine")
        if m in vt_machines:
            machine_rows[m].append(row)
        else:
            other_rows.append(row)

    resequenced_plan = []
    for m in vt_machines:
        rows = machine_rows.get(m, [])
        if len(rows) <= 1:
            resequenced_plan.extend(rows)
            continue

        yesterday_part = machine_last_part_yesterday.get(m)
        ordered   = []
        remaining = list(rows)

        seed = None
        if yesterday_part:
            for r in remaining:
                if r["Part"] == yesterday_part:
                    seed = r
                    break
        if seed is None:
            seed = max(remaining, key=lambda r: float(r.get("Priority_Score", 0) or 0))

        ordered.append(seed)
        remaining.remove(seed)

        while remaining:
            last_part  = ordered[-1]["Part"]
            last_color = part_color.get(last_part, "UNKNOWN")
            def _co_cost(r):
                p = r["Part"]
                c = part_color.get(p, "UNKNOWN")
                if p == last_part:
                    return -1.0
                base = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
                purge = (
                    COLOR_PURGE_HRS
                    if last_color not in ("UNKNOWN", "NEEDS_PURGE")
                    and c not in ("UNKNOWN",)
                    and last_color != c
                    else 0.0
                )
                return base + purge
            remaining.sort(key=_co_cost)
            ordered.append(remaining.pop(0))

        for i, row in enumerate(ordered):
            p = row["Part"]
            if i == 0:
                prev_part = yesterday_part
            else:
                prev_part = ordered[i - 1]["Part"]
            if prev_part is None or prev_part == p:
                new_co = 0.0
            else:
                base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
                prev_color = ALL_KNOWN_COLORS.get(prev_part, "UNKNOWN")
                new_color  = part_color.get(p, "UNKNOWN")
                purge = (
                    COLOR_PURGE_HRS
                    if prev_color not in ("UNKNOWN", "NEEDS_PURGE")
                    and new_color not in ("UNKNOWN",)
                    and prev_color != new_color
                    else 0.0
                )
                new_co = base_co + purge

            row["Changeover_Hrs"]  = round(new_co, 3)
            row["Changeover"]      = "No" if new_co == 0 else "Yes"
            row["Total_Hrs_Used"]  = round(
                new_co + float(row.get("Run_Hours", 0) or 0), 3
            )
            row["Color_Purge"] = "Yes" if (
                new_co > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001
            ) else "No"

        resequenced_plan.extend(ordered)

    resequenced_plan.extend(other_rows)
    return resequenced_plan

# =============================================================
# SECTION 15 — FIXED MACHINE SCHEDULING PASS
# =============================================================

def schedule_fixed_machines(machine_hours, machine_last_part,
                              current_inventory, plan, already_planned,
                              priority_scores, scenario_id):
    print(f"\n{'─'*65}")
    print(f"  FIXED MACHINE SCHEDULING PASS")
    print(f"  Buffer threshold: {SAFETY_DAYS} days")
    print(f"{'─'*65}")

    phase_a_machines = set()
    fixed_plan_rows  = []

    for machine, fixed_parts in sorted(machine_fixed_parts.items()):
        phase = fixed_machine_phase(machine, current_inventory)

        inv_summary = []
        for p in fixed_parts:
            daily    = indent_daily.get(p, 0)
            inv      = current_inventory.get(p, 0)
            days_cov = inv / daily if daily > 0 else 999
            r_val    = rate.get(p, 1)
            hrs_need = daily / r_val if r_val > 0 else 0
            inv_summary.append(
                f"{p}(inv={inv:.0f}={days_cov:.2f}d, need={hrs_need:.1f}h/d)"
            )
        print(f"\n  Machine: {machine}  Phase: {phase}")
        print(f"    Fixed parts: {' | '.join(inv_summary)}")

        if phase == "A":
            phase_a_machines.add(machine)

            chosen = pick_fixed_part_for_today(
                machine, current_inventory, machine_last_part
            )
            if chosen is None:
                print(f"    ⚠ No eligible fixed part — machine skipped")
                continue

            daily   = indent_daily.get(chosen, 0)
            r_val   = rate.get(chosen, 1)
            monthly = indent_monthly.get(chosen, 0)
            color   = part_color.get(chosen, "UNKNOWN")
            score   = priority_scores.get(chosen, 0)

            run_hrs_cap = fixed_part_run_hours(chosen, "A")
            co_hrs       = 0.0
            effective_run = run_hrs_cap
            qty = round(effective_run * r_val, 0)

            machine_hours[machine]      = round(effective_run, 4)
            current_inventory[chosen]   = round(
                current_inventory.get(chosen, 0) + qty, 0
            )
            machine_last_part[machine]  = chosen
            already_planned.add(chosen)

            row = {
                "Part":             chosen,
                "Color":            color,
                "Category":         part_category.get(chosen, "Runner"),
                "Fixed_Machine":    machine,
                "Fixed_Used":       "YES — Phase A (buffer building)",
                "Machine":          machine,
                "Run_Hours":        round(effective_run, 3),
                "Changeover_Hrs":   0.0,
                "Total_Hrs_Used":   round(effective_run, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(monthly, 0),
                "Daily_Indent":     round(daily, 2),
                "Today_Target":     round(today_target_qty.get(chosen, 0), 0),
                "Changeover":       "No",
                "Color_Purge":      "No",
                "Type":             f"Fixed-PhaseA [{run_hrs_cap}h cap]",
                "Role":             "Primary",
                "Tools_Available":  tools_available.get(chosen, 1),
                "Tools_Used":       1,
                "Runner_Lock":      "No",
                "Priority_Score":   score,
                "Phase":            1,
                "Indent_Met":       "YES" if qty >= daily else "NO — shortfall (normal for Phase A)",
                "Stagger_Adjusted": "No",
            }
            plan.append(row)
            fixed_plan_rows.append(row)

            inv_after  = current_inventory.get(chosen, 0)
            days_after = inv_after / daily if daily > 0 else 0
            print(f"    Phase A → {chosen}  {effective_run:.2f}h  "
                  f"qty={qty:.0f}  inv_after={inv_after:.0f} ({days_after:.2f}d)  "
                  f"{'23h' if run_hrs_cap == 23 else '22h'}")

        else:  # Phase B
            for chosen in fixed_parts:
                if machine_hours.get(machine, 0) >= AVAILABLE_HOURS - 0.05:
                    print(f"    Phase B → {chosen} SKIPPED — machine already at {machine_hours.get(machine,0):.2f}h")
                    already_planned.add(chosen)
                    continue

                daily   = indent_daily.get(chosen, 0)
                r_val   = rate.get(chosen, 1)
                monthly = indent_monthly.get(chosen, 0)
                color   = part_color.get(chosen, "UNKNOWN")
                score   = priority_scores.get(chosen, 0)

                min_run_for_indent = fixed_part_run_hours(chosen, "B")
                co_hrs        = 0.0

                available_now = round(AVAILABLE_HOURS - machine_hours.get(machine, 0), 4)
                effective_run = min(min_run_for_indent, available_now)
                if effective_run < MIN_RUN_HOURS:
                    print(f"    Phase B → {chosen} SKIPPED — only {available_now:.2f}h left, need {MIN_RUN_HOURS}h min")
                    already_planned.add(chosen)
                    continue

                inv_now_chosen = current_inventory.get(chosen, 0)
                cap_qty        = opd_cap(scenario_id) * daily
                headroom       = max(0.0, cap_qty - inv_now_chosen)

                if headroom > 0 and r_val > 0:
                    max_hrs_for_opd = headroom / r_val
                    effective_run   = min(available_now, max(effective_run, max_hrs_for_opd))
                    effective_run   = max(effective_run, MIN_RUN_HOURS)

                qty = round(effective_run * r_val, 0)

                machine_hours[machine]     = round(
                    machine_hours.get(machine, 0) + effective_run, 4
                )
                current_inventory[chosen]  = round(
                    current_inventory.get(chosen, 0) + qty, 0
                )
                machine_last_part[machine] = chosen
                already_planned.add(chosen)

                indent_met = qty >= (daily - 0.5)
                type_tag   = "Fixed-PhaseB [OPD build]" if effective_run > min_run_for_indent else "Fixed-PhaseB [min hours]"

                row = {
                    "Part":             chosen,
                    "Color":            color,
                    "Category":         part_category.get(chosen, "Runner"),
                    "Fixed_Machine":    machine,
                    "Fixed_Used":       "YES — Phase B (OPD build)",
                    "Machine":          machine,
                    "Run_Hours":        round(effective_run, 3),
                    "Changeover_Hrs":   0.0,
                    "Total_Hrs_Used":   round(effective_run, 3),
                    "Rate_Per_Hour":    round(r_val, 2),
                    "Production_Qty":   qty,
                    "Monthly_Indent":   round(monthly, 0),
                    "Daily_Indent":     round(daily, 2),
                    "Today_Target":     round(today_target_qty.get(chosen, 0), 0),
                    "Changeover":       "No",
                    "Color_Purge":      "No",
                    "Type":             type_tag,
                    "Role":             "Primary",
                    "Tools_Available":  tools_available.get(chosen, 1),
                    "Tools_Used":       1,
                    "Runner_Lock":      "No",
                    "Priority_Score":   score,
                    "Phase":            1,
                    "Indent_Met":       "YES" if indent_met else "NO — partial",
                    "Stagger_Adjusted": "No",
                }
                plan.append(row)
                fixed_plan_rows.append(row)

                remaining_hrs = round(AVAILABLE_HOURS - machine_hours.get(machine, 0), 4)
                print(f"    Phase B → {chosen}  {effective_run:.2f}h  "
                      f"qty={qty:.0f}  indent={'MET' if indent_met else 'PARTIAL'}  "
                      f"remaining={remaining_hrs:.2f}h (fixed-only, idle)")

    print(f"\n  Fixed machine pass: "
          f"{len(phase_a_machines)} Phase A (locked)  |  "
          f"{len(machine_fixed_parts) - len(phase_a_machines)} Phase B (partial free)")

    return phase_a_machines, fixed_plan_rows

# =============================================================
# SECTION 16 — TOOL-AWARE ASSIGNMENT
# =============================================================

def assign_part(part, scenario_id, machine_hours, machine_last_part,
                current_inventory, plan, already_planned,
                priority_scores):
    daily      = indent_daily.get(part, 0)
    monthly    = indent_monthly.get(part, 0)
    r_val      = rate.get(part, 1)
    inv_now    = current_inventory.get(part, 0)
    category   = part_category.get(part, "Stranger")
    tools      = tools_available.get(part, 1)
    score      = priority_scores.get(part, 0)
    color      = part_color.get(part, "UNKNOWN")
    inv_days   = inv_now / daily if daily > 0 else 999
    compatible = vt_compat.get(part, [])
    fixed_m    = part_fixed_machine.get(part)

    if not compatible:
        return []

    total_shortfall = max(0.0, daily - inv_now)
    hrs_for_full    = total_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS
    hrs_for_full    = max(MIN_RUN_HOURS, hrs_for_full)

    new_rows        = []
    produced_so_far = 0.0
    tools_used      = 0
    used_machines   = set()

    already_on_fixed = (
        fixed_m is not None
        and any(r["Machine"] == fixed_m and r["Part"] == part for r in plan)
    )

    if already_on_fixed:
        qty_from_fixed = sum(
            float(r["Production_Qty"])
            for r in plan
            if r["Part"] == part and r["Machine"] == fixed_m
        )
        produced_so_far = qty_from_fixed
        tools_used      = 1
        used_machines.add(fixed_m)

        indent_met_on_fixed = (produced_so_far >= total_shortfall - 0.5)

        if indent_met_on_fixed:
            already_planned.add(part)
            _do_inv_build(part, fixed_m, scenario_id, machine_hours,
                          current_inventory, plan, r_val, daily)
            return []
        else:
            if tools <= 1:
                already_planned.add(part)
                return []
    else:
        ranked, runner_lock = rank_machines(
            part, compatible, machine_hours, machine_last_part, inv_days
        )
        if not ranked:
            return []

        m1, co1, eff1, _ = ranked[0]
        run1 = min(eff1, hrs_for_full)
        run1 = max(run1, MIN_RUN_HOURS)
        qty1 = round(run1 * r_val, 0)

        fixed_used = (fixed_m is not None and m1 == fixed_m)

        machine_hours[m1]       = round(machine_hours.get(m1, 0) + co1 + run1, 4)
        current_inventory[part] = round(current_inventory.get(part, 0) + qty1, 0)
        machine_last_part[m1]   = part
        produced_so_far        += qty1
        tools_used             += 1
        used_machines.add(m1)
        already_planned.add(part)

        indent_met_on_primary = (produced_so_far >= total_shortfall - 0.5)

        last_m1       = machine_state.get(m1)
        purge_applied = (
            last_m1 is not None and last_m1 != part
            and ALL_KNOWN_COLORS.get(last_m1, "UNKNOWN") != color
            and ALL_KNOWN_COLORS.get(last_m1, "UNKNOWN") not in ("UNKNOWN", "NEEDS_PURGE")
            and color != "UNKNOWN"
        )

        type_tag = "Primary"
        if inv_now == 0:
            type_tag += " [ZERO-INV]"
        if fixed_used:
            type_tag += " [FIXED-MACHINE]"
        elif fixed_m is not None:
            type_tag += " [FIXED-FALLBACK]"

        new_rows.append({
            "Part":             part,
            "Color":            color,
            "Category":         category,
            "Fixed_Machine":    fixed_m or "—",
            "Fixed_Used":       "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
            "Machine":          m1,
            "Run_Hours":        round(run1, 3),
            "Changeover_Hrs":   round(co1, 3),
            "Total_Hrs_Used":   round(co1 + run1, 3),
            "Rate_Per_Hour":    round(r_val, 2),
            "Production_Qty":   qty1,
            "Monthly_Indent":   round(monthly, 0),
            "Daily_Indent":     round(daily, 2),
            "Today_Target":     round(today_target_qty.get(part, 0), 0),
            "Changeover":       "No" if co1 == 0 else "Yes",
            "Color_Purge":      "Yes" if purge_applied else "No",
            "Type":             type_tag,
            "Role":             "Primary",
            "Tools_Available":  tools,
            "Tools_Used":       1,
            "Runner_Lock":      "YES" if runner_lock else "No",
            "Priority_Score":   score,
            "Phase":            1,
            "Indent_Met":       "YES" if indent_met_on_primary else "NO — shortfall remains",
            "Stagger_Adjusted": "No",
        })

        if indent_met_on_primary:
            _do_inv_build(part, m1, scenario_id, machine_hours,
                          current_inventory, new_rows, r_val, daily)
            for row in new_rows:
                row["Tools_Used"] = tools_used
            return new_rows

    # Tool expansion — only for Runners with tools > 1 and shortfall remaining
    is_critical = (inv_now == 0)

    if category in ("Stranger", "Repeater"):
        tool_hard_cap = 1
    else:
        if is_critical:
            tool_hard_cap = tools
        else:
            tool_hard_cap = min(2, tools)

    if tool_hard_cap <= 1 and tools_used >= 1:
        if new_rows:
            _do_inv_build(part, new_rows[0]["Machine"], scenario_id, machine_hours,
                          current_inventory, new_rows, r_val, daily)
        for row in new_rows:
            row["Tools_Used"] = tools_used
        return new_rows

    while (produced_so_far < (total_shortfall - 0.5)
           and tools_used < tool_hard_cap):
        shortfall_now = total_shortfall - produced_so_far
        hrs_needed    = max(
            MIN_RUN_HOURS,
            shortfall_now / r_val if r_val > 0 else MIN_RUN_HOURS,
        )

        remaining_machines = [
            m for m in compatible
            if m not in used_machines
            and m not in machine_fixed_parts
        ]
        ranked_next, _ = rank_machines(
            part, remaining_machines, machine_hours,
            machine_last_part, inv_days
        )
        if not ranked_next:
            break

        mx, cox, effx, _ = ranked_next[0]
        run_x = min(effx, hrs_needed)
        run_x = max(run_x, MIN_RUN_HOURS)
        qty_x = round(run_x * r_val, 0)

        machine_hours[mx]       = round(machine_hours.get(mx, 0) + cox + run_x, 4)
        current_inventory[part] = round(current_inventory.get(part, 0) + qty_x, 0)
        machine_last_part[mx]   = part
        produced_so_far        += qty_x
        tools_used             += 1
        used_machines.add(mx)

        indent_met_here = (produced_so_far >= total_shortfall - 0.5)

        last_mx = machine_state.get(mx)
        purge_x = (
            last_mx is not None and last_mx != part
            and ALL_KNOWN_COLORS.get(last_mx, "UNKNOWN") != color
            and ALL_KNOWN_COLORS.get(last_mx, "UNKNOWN") not in ("UNKNOWN", "NEEDS_PURGE")
            and color != "UNKNOWN"
        )

        new_rows.append({
            "Part":             part,
            "Color":            color,
            "Category":         category,
            "Fixed_Machine":    fixed_m or "—",
            "Fixed_Used":       "N/A — expansion",
            "Machine":          mx,
            "Run_Hours":        round(run_x, 3),
            "Changeover_Hrs":   round(cox, 3),
            "Total_Hrs_Used":   round(cox + run_x, 3),
            "Rate_Per_Hour":    round(r_val, 2),
            "Production_Qty":   qty_x,
            "Monthly_Indent":   round(monthly, 0),
            "Daily_Indent":     round(daily, 2),
            "Today_Target":     round(today_target_qty.get(part, 0), 0),
            "Changeover":       "No" if cox == 0 else "Yes",
            "Color_Purge":      "Yes" if purge_x else "No",
            "Type":             "Tool-Expansion",
            "Role":             f"Tool-Expansion (tool {tools_used}/{tools})",
            "Tools_Available":  tools,
            "Tools_Used":       tools_used,
            "Runner_Lock":      "No",
            "Priority_Score":   score,
            "Phase":            2,
            "Indent_Met":       "YES" if indent_met_here else "NO — shortfall remains",
            "Stagger_Adjusted": "No",
        })
        crisis_tag = " [CRISIS — 3rd+ tool]" if tools_used >= 3 else ""
        print(f"      ↳ TOOL-EXP {part:26s} tool {tools_used}/{tool_hard_cap} → "
              f"{mx:15s}  {run_x:.2f}h  qty={qty_x:.0f}{crisis_tag}")

    if new_rows:
        primary_machine = new_rows[0]["Machine"]
        _do_inv_build(part, primary_machine, scenario_id, machine_hours,
                      current_inventory, new_rows, r_val, daily)

    for row in new_rows:
        row["Tools_Used"] = tools_used

    return new_rows


def _do_inv_build(part, machine, scenario_id, machine_hours,
                  current_inventory, rows_to_extend, r_val, daily):
    if machine in _phase_a_machines:
        return
    cap_days     = opd_cap(scenario_id)
    inv_after    = current_inventory.get(part, 0)
    cap_qty      = cap_days * daily
    headroom_qty = max(0.0, cap_qty - inv_after)
    if headroom_qty <= 0:
        return

    target_row = None
    if isinstance(rows_to_extend, list):
        for row in rows_to_extend:
            if isinstance(row, dict) and row.get("Part") == part and row.get("Machine") == machine:
                target_row = row
                break
    if target_row is None:
        return

    free_m = round(AVAILABLE_HOURS - machine_hours.get(machine, 0), 4)
    if free_m < 0.05:
        return
    extend_hrs = min(free_m, headroom_qty / r_val if r_val > 0 else 0)
    if extend_hrs < 0.05:
        return
    extra_qty = round(extend_hrs * r_val, 0)
    target_row["Run_Hours"]      = round(float(target_row["Run_Hours"]) + extend_hrs, 3)
    target_row["Total_Hrs_Used"] = round(
        float(target_row["Changeover_Hrs"]) + float(target_row["Run_Hours"]), 3
    )
    target_row["Production_Qty"] = round(
        float(target_row["Production_Qty"]) + extra_qty, 0
    )
    target_row["Type"] = str(target_row["Type"]) + "+InvBuild"
    machine_hours[machine]       = round(machine_hours.get(machine, 0) + extend_hrs, 4)
    current_inventory[part]      = round(current_inventory.get(part, 0) + extra_qty, 0)
    print(f"      ↳ INV-BUILD {part:25s} on {machine:15s}  "
          f"+{extend_hrs:.2f}h  qty+={extra_qty:.0f}")

# =============================================================
# SECTION 17 — RUNNER PRIORITY ENFORCEMENT
# =============================================================

def enforce_runner_priority(plan, machine_hours, machine_last_part,
                             current_inventory, already_planned,
                             priority_scores, inventory_start_of_day):
    print(f"\n{'─'*65}")
    print(f"  RUNNER PRIORITY ENFORCEMENT (threshold: < {RUNNER_PRIORITY_DAYS} days)")
    print(f"{'─'*65}")

    runner_priority_log = []

    critical_runners = []
    for part in vt_compat:
        if part in already_planned:
            continue
        if part_category.get(part, "Stranger") != "Runner":
            continue
        daily = indent_daily.get(part, 0)
        r_val = rate.get(part, 1)
        if daily <= 0 or r_val <= 0:
            continue
        inv_now  = current_inventory.get(part, 0)
        days_now = inv_now / daily if daily > 0 else 999
        if days_now >= RUNNER_PRIORITY_DAYS:
            continue
        skip, _ = should_skip(part)
        if skip:
            continue
        if not vt_compat.get(part):
            continue
        critical_runners.append(part)

    if not critical_runners:
        print(f"  No critical unplanned Runners found  ✓")
        return runner_priority_log

    critical_runners.sort(key=lambda p: indent_daily.get(p, 0), reverse=True)
    print(f"  Critical Runners to enforce: {len(critical_runners)}")

    for runner in critical_runners:
        r_daily    = indent_daily.get(runner, 0)
        r_inv      = current_inventory.get(runner, 0)
        r_inv_sod  = inventory_start_of_day.get(runner, r_inv)
        r_rate     = rate.get(runner, 1)
        r_color    = part_color.get(runner, "UNKNOWN")
        r_score    = priority_scores.get(runner, 0)
        r_monthly  = indent_monthly.get(runner, 0)
        fixed_m    = part_fixed_machine.get(runner)
        r_days_now = r_inv / r_daily if r_daily > 0 else 0
        r_days_sod = r_inv_sod / r_daily if r_daily > 0 else 0

        shortfall_qty = max(0.0, r_daily - r_inv)
        hours_needed  = max(
            MIN_RUN_HOURS,
            shortfall_qty / r_rate if r_rate > 0 else MIN_RUN_HOURS,
        )

        machines_runner_on = {row["Machine"] for row in plan if row["Part"] == runner}
        if len(machines_runner_on) >= tools_available.get(runner, 1):
            print(f"    ✗ {runner}  — all {tools_available.get(runner,1)} tool(s) already deployed")
            runner_priority_log.append({
                "Runner_Part": runner, "Runner_Days_SOD": round(r_days_sod, 2),
                "Runner_Days_Now": round(r_days_now, 2), "Fixed_Machine": fixed_m or "—",
                "Fixed_Used": "FAILED — tool cap", "Runner_Daily_Indent": round(r_daily, 2),
                "Runner_Inv_Before_SOD": round(r_inv_sod, 0), "Runner_Shortfall": round(shortfall_qty, 0),
                "Machine_Assigned": "—", "Hours_Needed": round(hours_needed, 3),
                "Hours_Assigned": 0, "Qty_Produced": 0, "Displacement_Used": "N/A",
                "Victims": "—", "Total_Hours_Reclaimed": "—",
                "Result": f"FAILED — all {tools_available.get(runner,1)} tool(s) deployed",
            })
            continue

        compatible_machines = [
            m for m in vt_compat.get(runner, [])
            if m not in machine_fixed_parts
        ]
        if fixed_m and fixed_m not in compatible_machines:
            compatible_machines = [fixed_m] + compatible_machines

        if fixed_m and fixed_m in compatible_machines:
            ordered_machines = [fixed_m] + [m for m in compatible_machines if m != fixed_m]
        else:
            ordered_machines = compatible_machines

        print(f"\n  Runner: {runner}  sod_days={r_days_sod:.2f}  now_days={r_days_now:.2f}  "
              f"inv={r_inv:.0f}  daily={r_daily:.2f}  need={hours_needed:.2f}h")

        best_machine    = None
        best_co_hrs     = 0.0
        best_victims    = []
        best_disruption = float("inf")

        for m in ordered_machines:
            co_hrs   = _co_hrs_for(runner, m, machine_last_part)
            used_hrs = machine_hours.get(m, 0)
            free_hrs = round(AVAILABLE_HOURS - used_hrs, 4)
            eff_free = round(free_hrs - co_hrs, 4)

            if eff_free >= hours_needed:
                run_h = max(MIN_RUN_HOURS, min(eff_free, hours_needed))
                qty   = round(run_h * r_rate, 0)
                fixed_used = (fixed_m is not None and m == fixed_m)

                machine_hours[m]          = round(used_hrs + co_hrs + run_h, 4)
                current_inventory[runner] = round(current_inventory.get(runner, 0) + qty, 0)
                machine_last_part[m]      = runner
                already_planned.add(runner)

                purge  = co_hrs > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001

                plan.append({
                    "Part": runner, "Color": r_color, "Category": "Runner",
                    "Fixed_Machine": fixed_m or "—",
                    "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
                    "Machine": m, "Run_Hours": round(run_h, 3),
                    "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_h, 3),
                    "Rate_Per_Hour": round(r_rate, 2), "Production_Qty": qty,
                    "Monthly_Indent": round(r_monthly, 0), "Daily_Indent": round(r_daily, 2),
                    "Today_Target": round(today_target_qty.get(runner, 0), 0),
                    "Changeover": "No" if co_hrs == 0 else "Yes",
                    "Color_Purge": "Yes" if purge else "No",
                    "Type": "Runner-Priority [<2-DAY] (free capacity)" + (" [FIXED]" if fixed_used else ""),
                    "Role": "Primary", "Tools_Available": tools_available.get(runner, 1),
                    "Tools_Used": 1, "Runner_Lock": "No", "Priority_Score": r_score,
                    "Phase": 1, "Indent_Met": "YES" if qty >= shortfall_qty else "NO — partial",
                    "Stagger_Adjusted": "No",
                })

                runner_priority_log.append({
                    "Runner_Part": runner, "Runner_Days_SOD": round(r_days_sod, 2),
                    "Runner_Days_Now": round(r_days_now, 2), "Fixed_Machine": fixed_m or "—",
                    "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
                    "Runner_Daily_Indent": round(r_daily, 2),
                    "Runner_Inv_Before_SOD": round(r_inv_sod, 0),
                    "Runner_Shortfall": round(shortfall_qty, 0),
                    "Machine_Assigned": m, "Hours_Needed": round(hours_needed, 3),
                    "Hours_Assigned": round(run_h, 3), "Qty_Produced": qty,
                    "Displacement_Used": "No — free capacity", "Victims": "—",
                    "Total_Hours_Reclaimed": "—", "Result": "PLANNED — free capacity",
                })
                print(f"    → {runner} placed on {m} (free capacity)  run={run_h:.2f}h  qty={qty:.0f}  ✓")
                best_machine = "DONE"
                break

            # Displacement check
            machine_rows = [r for r in plan if r["Machine"] == m]
            yieldable = [
                r for r in machine_rows
                if part_category.get(r["Part"], "Stranger") in ("Stranger", "Repeater")
                and (
                    current_inventory.get(r["Part"], 0) /
                    indent_daily.get(r["Part"], 1)
                    if indent_daily.get(r["Part"], 0) > 0 else 999
                ) > r_days_now
            ]
            if not yieldable:
                continue

            yieldable.sort(key=lambda r: (
                indent_daily.get(r["Part"], 0),
                -float(r.get("Production_Qty", 0)),
            ))

            reclaimable_detail = []
            for row in yieldable:
                row_run = float(row.get("Run_Hours", 0))
                if row_run <= 0:
                    continue
                if row_run > MIN_RUN_HOURS:
                    reclaimable_detail.append(
                        (row, round(row_run - MIN_RUN_HOURS, 4), "partial")
                    )
                else:
                    reclaimable_detail.append(
                        (row, round(row_run, 4), "full_remove")
                    )

            total_reclaimable        = sum(x[1] for x in reclaimable_detail)
            runner_eff_after_reclaim = round(free_hrs + total_reclaimable - co_hrs, 4)

            if runner_eff_after_reclaim < max(MIN_RUN_HOURS, hours_needed):
                continue

            disruption = total_reclaimable
            if disruption < best_disruption:
                best_disruption = disruption
                best_machine    = m
                best_co_hrs     = co_hrs
                best_victims    = reclaimable_detail

        if best_machine is None:
            print(f"    ✗ {runner}  — no machine qualifies")
            runner_priority_log.append({
                "Runner_Part": runner, "Runner_Days_SOD": round(r_days_sod, 2),
                "Runner_Days_Now": round(r_days_now, 2), "Fixed_Machine": fixed_m or "—",
                "Fixed_Used": "FAILED", "Runner_Daily_Indent": round(r_daily, 2),
                "Runner_Inv_Before_SOD": round(r_inv_sod, 0),
                "Runner_Shortfall": round(shortfall_qty, 0),
                "Machine_Assigned": "—", "Hours_Needed": round(hours_needed, 3),
                "Hours_Assigned": 0, "Qty_Produced": 0, "Displacement_Used": "N/A",
                "Victims": "—", "Total_Hours_Reclaimed": "—",
                "Result": "FAILED — no eligible machine",
            })
            continue

        if best_machine == "DONE":
            continue

        # Carve hours from victims
        hours_to_free    = hours_needed
        victim_log_parts = []
        total_reclaimed  = 0.0

        for (victim_row, reclaimable_hrs, reclaim_type) in best_victims:
            if hours_to_free <= 0.001:
                break
            vpart      = victim_row["Part"]
            v_rate     = rate.get(vpart, 1)
            v_run_orig = float(victim_row.get("Run_Hours", 0))
            carve_hrs  = round(min(reclaimable_hrs, hours_to_free), 4)
            if carve_hrs <= 0:
                continue

            new_run_hrs = round(v_run_orig - carve_hrs, 4)

            if new_run_hrs < MIN_RUN_HOURS:
                plan.remove(victim_row)
                machine_hours[best_machine] = round(
                    machine_hours.get(best_machine, 0) - v_run_orig, 4
                )
                lost_qty = round(v_run_orig * v_rate, 0)
                current_inventory[vpart] = round(
                    current_inventory.get(vpart, 0) - lost_qty, 0
                )
                actually_freed = v_run_orig
                victim_log_parts.append(f"{vpart} REMOVED ({v_run_orig:.2f}h / {lost_qty:.0f} pcs)")
                print(f"    ↳ YIELD (remove) {vpart:28s}  freed {v_run_orig:.2f}h")
            else:
                lost_qty  = round(carve_hrs * v_rate, 0)
                new_qty   = round(new_run_hrs * v_rate, 0)
                new_total = round(
                    float(victim_row.get("Changeover_Hrs", 0)) + new_run_hrs, 3
                )
                victim_row["Run_Hours"]      = new_run_hrs
                victim_row["Production_Qty"] = new_qty
                victim_row["Total_Hrs_Used"] = new_total
                victim_row["Type"]           = str(victim_row.get("Type", "Primary")) + " [YIELDED_TO_RUNNER]"
                machine_hours[best_machine] = round(
                    machine_hours.get(best_machine, 0) - carve_hrs, 4
                )
                current_inventory[vpart] = round(
                    current_inventory.get(vpart, 0) - lost_qty, 0
                )
                actually_freed = carve_hrs
                victim_log_parts.append(f"{vpart} −{carve_hrs:.2f}h")
                print(f"    ↳ YIELD (reduce) {vpart:28s}  −{carve_hrs:.2f}h  −{lost_qty:.0f} pcs")

            hours_to_free   = round(hours_to_free - actually_freed, 4)
            total_reclaimed = round(total_reclaimed + actually_freed, 4)

        used_now = machine_hours.get(best_machine, 0)
        free_now = round(AVAILABLE_HOURS - used_now, 4)
        eff_free = round(free_now - best_co_hrs, 4)

        if eff_free < MIN_RUN_HOURS:
            print(f"    ✗ {runner}  safety check failed after carve → skip")
            runner_priority_log.append({
                "Runner_Part": runner, "Runner_Days_SOD": round(r_days_sod, 2),
                "Runner_Days_Now": round(r_days_now, 2), "Fixed_Machine": fixed_m or "—",
                "Fixed_Used": "FAILED", "Runner_Daily_Indent": round(r_daily, 2),
                "Runner_Inv_Before_SOD": round(r_inv_sod, 0),
                "Runner_Shortfall": round(shortfall_qty, 0),
                "Machine_Assigned": best_machine, "Hours_Needed": round(hours_needed, 3),
                "Hours_Assigned": 0, "Qty_Produced": 0, "Displacement_Used": "Yes",
                "Victims": "; ".join(victim_log_parts),
                "Total_Hours_Reclaimed": round(total_reclaimed, 3),
                "Result": "FAILED — safety check after carve",
            })
            continue

        run_hrs    = max(MIN_RUN_HOURS, min(eff_free, hours_needed))
        qty        = round(run_hrs * r_rate, 0)
        fixed_used = (fixed_m is not None and best_machine == fixed_m)
        purge      = best_co_hrs > vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS) + 0.001

        machine_hours[best_machine]     = round(used_now + best_co_hrs + run_hrs, 4)
        current_inventory[runner]       = round(current_inventory.get(runner, 0) + qty, 0)
        machine_last_part[best_machine] = runner
        already_planned.add(runner)

        plan.append({
            "Part": runner, "Color": r_color, "Category": "Runner",
            "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
            "Machine": best_machine, "Run_Hours": round(run_hrs, 3),
            "Changeover_Hrs": round(best_co_hrs, 3),
            "Total_Hrs_Used": round(best_co_hrs + run_hrs, 3),
            "Rate_Per_Hour": round(r_rate, 2), "Production_Qty": qty,
            "Monthly_Indent": round(r_monthly, 0), "Daily_Indent": round(r_daily, 2),
            "Today_Target": round(today_target_qty.get(runner, 0), 0),
            "Changeover": "No" if best_co_hrs == 0 else "Yes",
            "Color_Purge": "Yes" if purge else "No",
            "Type": "Runner-Priority [<2-DAY] [DISPLACED]" + (" [FIXED]" if fixed_used else ""),
            "Role": "Primary", "Tools_Available": tools_available.get(runner, 1),
            "Tools_Used": 1, "Runner_Lock": "No", "Priority_Score": r_score,
            "Phase": 1, "Indent_Met": "YES" if qty >= shortfall_qty else "NO — partial",
            "Stagger_Adjusted": "No",
        })

        runner_priority_log.append({
            "Runner_Part": runner, "Runner_Days_SOD": round(r_days_sod, 2),
            "Runner_Days_Now": round(r_days_now, 2), "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
            "Runner_Daily_Indent": round(r_daily, 2),
            "Runner_Inv_Before_SOD": round(r_inv_sod, 0),
            "Runner_Shortfall": round(shortfall_qty, 0),
            "Machine_Assigned": best_machine, "Hours_Needed": round(hours_needed, 3),
            "Hours_Assigned": round(run_hrs, 3), "Qty_Produced": qty,
            "Displacement_Used": "Yes", "Victims": "; ".join(victim_log_parts),
            "Total_Hours_Reclaimed": round(total_reclaimed, 3),
            "Result": ("PLANNED — displacement successful" if qty >= shortfall_qty - 0.5
                       else "PLANNED — partial (machine hours limited)"),
        })

        print(f"    ✓ {runner:30s} → {best_machine}  run={run_hrs:.2f}h  "
              f"qty={qty:.0f}  reclaimed={total_reclaimed:.2f}h")

    total_placed = sum(1 for r in runner_priority_log if "PLANNED" in r.get("Result", ""))
    total_failed = sum(1 for r in runner_priority_log if "FAILED" in r.get("Result", ""))
    print(f"\n  Runner Priority complete: {total_placed} placed  |  {total_failed} failed")

    return runner_priority_log

# =============================================================
# SECTION 18 — DISPLACEMENT PRE-PASS
# =============================================================

def displace_for_zero_inv(part, machine_hours, machine_last_part,
                           current_inventory, plan, already_planned,
                           priority_scores):
    daily      = indent_daily.get(part, 0)
    r_val      = rate.get(part, 1)
    category   = part_category.get(part, "Stranger")
    score      = priority_scores.get(part, 0)
    compatible = vt_compat.get(part, [])
    fixed_m    = part_fixed_machine.get(part)

    if not compatible:
        return False

    candidate_machines = [
        m for m in compatible
        if m not in _phase_a_machines
        and round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4) < MIN_RUN_HOURS
    ]
    if not candidate_machines:
        return False

    best_machine     = None
    best_victim_row  = None
    best_victim_days = -1

    part_days_cov = (
        current_inventory.get(part, 0) / daily if daily > 0 else 0
    )

    for m in candidate_machines:
        for row in [r for r in plan if r["Machine"] == m]:
            vpart  = row["Part"]
            vdaily = indent_daily.get(vpart, 0)
            vinv   = current_inventory.get(vpart, 0)
            vdays  = vinv / vdaily if vdaily > 0 else 999
            if vdays <= part_days_cov:
                continue
            if vdays < SAFETY_DAYS or vinv <= 0:
                continue
            vrun = float(row.get("Run_Hours", 0))
            if vrun - MIN_RUN_HOURS < MIN_RUN_HOURS:
                continue
            if vdays > best_victim_days:
                best_victim_days = vdays
                best_victim_row  = row
                best_machine     = m

    if best_machine is None or best_victim_row is None:
        return False

    vpart    = best_victim_row["Part"]
    vr_val   = rate.get(vpart, 1)
    reduce_h = MIN_RUN_HOURS
    lost_qty = round(reduce_h * vr_val, 0)

    best_victim_row["Run_Hours"]      = round(float(best_victim_row["Run_Hours"]) - reduce_h, 3)
    best_victim_row["Production_Qty"] = round(float(best_victim_row["Production_Qty"]) - lost_qty, 0)
    best_victim_row["Total_Hrs_Used"] = round(
        float(best_victim_row.get("Changeover_Hrs", 0)) + float(best_victim_row["Run_Hours"]), 3)
    best_victim_row["Type"] = str(best_victim_row.get("Type", "Primary")) + " [DISPLACED]"

    current_inventory[vpart]    = round(current_inventory.get(vpart, 0) - lost_qty, 0)
    machine_hours[best_machine] = round(machine_hours.get(best_machine, 0) - reduce_h, 4)

    p_col = part_color.get(part, "UNKNOWN")
    last  = machine_last_part.get(best_machine)
    l_col = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
    if last is None or last == part:
        co_hrs = 0.0
    else:
        base_co = vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS)
        purge   = (
            COLOR_PURGE_HRS
            if p_col != l_col and p_col != "UNKNOWN" and l_col not in ("UNKNOWN", "NEEDS_PURGE")
            else 0.0
        )
        co_hrs = base_co + purge

    eff_free = round(AVAILABLE_HOURS - machine_hours.get(best_machine, 0) - co_hrs, 4)
    hrs_for_indent = max(MIN_RUN_HOURS, daily / r_val if r_val > 0 else MIN_RUN_HOURS)
    run_hrs = max(MIN_RUN_HOURS, min(eff_free, hrs_for_indent))
    qty     = round(run_hrs * r_val, 0)

    machine_hours[best_machine]     = round(
        machine_hours.get(best_machine, 0) + co_hrs + run_hrs, 4)
    current_inventory[part]         = round(current_inventory.get(part, 0) + qty, 0)
    machine_last_part[best_machine] = part
    already_planned.add(part)

    has_purge  = co_hrs > vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS) + 0.001
    fixed_used = (fixed_m is not None and best_machine == fixed_m)

    plan.append({
        "Part": part, "Color": p_col, "Category": category,
        "Fixed_Machine": fixed_m or "—",
        "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
        "Machine": best_machine, "Run_Hours": round(run_hrs, 3),
        "Changeover_Hrs": round(co_hrs, 3),
        "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty,
        "Monthly_Indent": round(indent_monthly.get(part, 0), 0),
        "Daily_Indent": round(daily, 2),
        "Today_Target": round(today_target_qty.get(part, 0), 0),
        "Changeover": "No" if co_hrs == 0 else "Yes",
        "Color_Purge": "Yes" if has_purge else "No",
        "Type": "Displacement [ZERO-INV PRIORITY]",
        "Role": "Primary", "Tools_Available": tools_available.get(part, 1),
        "Tools_Used": 1, "Runner_Lock": "No", "Priority_Score": score,
        "Phase": 1, "Indent_Met": "YES" if qty >= daily else "NO — partial",
        "Stagger_Adjusted": "No",
    })

    print(f"      ↳ DISPLACEMENT  {part:26s} → {best_machine:15s}  "
          f"freed from {vpart} ({best_victim_days:.1f}d)  "
          f"run={run_hrs:.2f}h  qty={qty:.0f}")
    return True

# =============================================================
# SECTION 19 — TOOL-CHANGER HELPERS
# =============================================================

def _fmt_h(h):
    try:
        total_min = int(round(float(h) * 60))
        return f"{total_min // 60:02d}:{total_min % 60:02d}"
    except Exception:
        return "??"


def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours")       or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine":       m,
                    "part_before":   m_rows[i - 1]["Part"],
                    "part_after":    row["Part"],
                    "co_duration":   co_h,
                    "natural_start": cursor,
                    "row_before":    m_rows[i - 1],
                    "row_after":     row,
                    "actual_start":  None,
                    "wait_hrs":      0.0,
                })
            cursor += co_h + run_h
    return events


def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += (float(r.get("Changeover_Hrs") or 0)
                   + float(r.get("Run_Hours") or 0))
    return cursor


def _machine_spare(m, plan):
    used = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == m
    )
    return max(0.0, AVAILABLE_HOURS - used)


def _extend_row_before(ev, wait_hrs, plan, machine_hours):
    m         = ev["machine"]
    spare     = _machine_spare(m, plan)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(
        float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3
    )
    rb["Stagger_Adjusted"] = (
        f"CO: extended +{round(extend_by * 60, 1)}min to fill TC wait"
    )
    machine_hours[m] = round(machine_hours.get(m, 0) + extend_by, 4)
    return extend_by, extra

# =============================================================
# SECTION 20 — CO STAGGER
# =============================================================

MIN_CO_GAP_HRS = 20 / 60.0


def _finish_time_of_co(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    total = 0.0
    for row in plan:
        if row["Machine"] != m:
            continue
        if row is target:
            break
        total += float(row.get("Run_Hours") or 0)
        total += float(row.get("Changeover_Hrs") or 0)
    return round(total, 4)


def stagger_co_by_quantity(plan, machines, scenario_id, current_inventory):
    events = _collect_co_events(plan, machines)
    if len(events) < 2:
        return 0

    total_co_in_plan = len(events)
    if total_co_in_plan > MAX_DAILY_CO:
        print(f"    [CO-CAP] {total_co_in_plan} COs in plan > MAX_DAILY_CO={MAX_DAILY_CO}")

    adjustments = 0
    max_passes  = len(events) * 2

    for _ in range(max_passes):
        for ev in events:
            ev["_ft"] = _finish_time_of_co(ev, plan)
        events.sort(key=lambda e: e["_ft"])

        conflict = None
        for i in range(len(events) - 1):
            co_dur_e = events[i]["co_duration"]
            required = co_dur_e + MIN_CO_GAP_HRS
            gap      = events[i + 1]["_ft"] - events[i]["_ft"]
            if gap < required - 0.001:
                conflict = (events[i], events[i + 1], gap, required)
                break

        if conflict is None:
            break

        ev_early, ev_late, gap, required_gap = conflict
        shortfall_hrs = required_gap - gap

        row_late   = ev_late["row_before"]
        p_late     = row_late["Part"]
        r_late     = rate.get(p_late, 1)
        m_late     = ev_late["machine"]
        daily_l    = indent_daily.get(p_late, 0)
        inv_l      = current_inventory.get(p_late, 0)
        cap_qty    = opd_cap(scenario_id) * daily_l
        produced_l = float(row_late.get("Production_Qty") or 0)
        headroom   = max(0.0, cap_qty - inv_l)
        push_hrs   = shortfall_hrs
        push_qty   = round(push_hrs * r_late, 0)
        used_m     = sum(
            float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
            for r in plan if r["Machine"] == m_late
        )
        free_m   = max(0.0, AVAILABLE_HOURS - used_m)
        can_push = (
            push_qty <= headroom
            and push_hrs <= free_m + 0.001
            and r_late > 0
            and m_late not in _phase_a_machines
        )

        if can_push:
            row_late["Run_Hours"]      = round(float(row_late.get("Run_Hours") or 0) + push_hrs, 3)
            row_late["Production_Qty"] = round(produced_l + push_qty, 0)
            row_late["Total_Hrs_Used"] = round(
                float(row_late.get("Changeover_Hrs") or 0) + float(row_late["Run_Hours"]), 3)
            row_late["Stagger_Adjusted"] = f"CO-stagger PUSH +{round(push_hrs * 60, 1)}min"
            current_inventory[p_late] = round(current_inventory.get(p_late, 0) + push_qty, 0)
            adjustments += 1
            continue

        row_early  = ev_early["row_before"]
        p_early    = row_early["Part"]
        r_early    = rate.get(p_early, 1)
        daily_e    = indent_daily.get(p_early, 0)
        inv_e      = current_inventory.get(p_early, 0)
        produced_e = float(row_early.get("Production_Qty") or 0)
        min_qty_e  = max(0.0, daily_e - inv_e)
        max_pull   = max(0.0, produced_e - min_qty_e)
        pull_hrs   = shortfall_hrs
        pull_qty   = round(pull_hrs * r_early, 0)
        can_pull   = (
            pull_qty <= max_pull
            and r_early > 0
            and produced_e - pull_qty >= MIN_RUN_HOURS * r_early
            and ev_early["machine"] not in _phase_a_machines
        )

        if can_pull:
            row_early["Run_Hours"]      = round(float(row_early.get("Run_Hours") or 0) - pull_hrs, 3)
            row_early["Production_Qty"] = round(produced_e - pull_qty, 0)
            row_early["Total_Hrs_Used"] = round(
                float(row_early.get("Changeover_Hrs") or 0) + float(row_early["Run_Hours"]), 3)
            row_early["Stagger_Adjusted"] = f"CO-stagger PULL -{round(pull_hrs * 60, 1)}min"
            current_inventory[p_early] = round(current_inventory.get(p_early, 0) - pull_qty, 0)
            adjustments += 1
            continue

        print(f"    [CO-STAGGER SKIP]  {ev_early['machine']} / {ev_late['machine']}  unresolvable")
        ev_late["_ft"] = ev_early["_ft"] + MIN_CO_GAP_HRS
        break

    return adjustments


def stagger_changeovers_serial_queue(plan, machines, machine_hours):
    print(f"\n  Tool-Changer Serial Queue Scheduler  (max {MAX_DAILY_CO} COs/day)")

    events = _collect_co_events(plan, machines)
    if not events:
        print(f"  No changeovers in plan — tool changer idle  ✓")
        return

    events.sort(key=lambda e: e["natural_start"])
    print(f"  {len(events)} CO events across "
          f"{len({e['machine'] for e in events})} machines")

    if len(events) > MAX_DAILY_CO:
        print(f"  ⚠ WARNING: {len(events)} COs exceed MAX_DAILY_CO={MAX_DAILY_CO}")
        events = events[:MAX_DAILY_CO]

    print(f"\n  {'#':<4} {'Machine':<18} {'Part Before':<22} {'Part After':<22} "
          f"{'Dur':>5} {'Natural':>8} {'Actual':>8} {'Wait':>7} {'Fill':>6} {'Purge':>6}")
    print(f"  {'─'*105}")

    tool_changer_free_at = 0.0
    total_extra_pcs      = 0
    total_wait_min       = 0.0

    for idx, ev in enumerate(events, 1):
        natural_start        = _recompute_natural_start(ev, plan)
        co_h                 = ev["co_duration"]
        actual_start         = max(natural_start, tool_changer_free_at)
        wait_hrs             = round(actual_start - natural_start, 4)
        tool_changer_free_at = actual_start + co_h

        extra_pcs = 0
        if wait_hrs > 0.001:
            _, extra_pcs     = _extend_row_before(ev, wait_hrs, plan, machine_hours)
            total_extra_pcs += extra_pcs
            total_wait_min  += wait_hrs * 60

        ev["actual_start"] = actual_start
        ev["wait_hrs"]     = wait_hrs

        wait_str     = f"+{round(wait_hrs * 60, 1)}m" if wait_hrs > 0.001 else "none"
        fill_str     = f"+{extra_pcs:.0f}" if extra_pcs > 0 else "—"
        before_color = part_color.get(ev["part_before"], "?")
        after_color  = part_color.get(ev["part_after"],  "?")
        purge_str    = (
            "PURGE"
            if before_color != after_color
            and before_color not in ("?", "UNKNOWN")
            and after_color  not in ("?", "UNKNOWN")
            else "—"
        )

        print(f"  {idx:<4} {ev['machine']:<18} {ev['part_before']:<22} "
              f"{ev['part_after']:<22} "
              f"{round(co_h * 60, 1):>4.0f}m "
              f"{_fmt_h(natural_start):>8} "
              f"{_fmt_h(actual_start):>8} "
              f"{wait_str:>7} "
              f"{fill_str:>6} "
              f"{purge_str:>6}")

    n_waited = sum(1 for e in events if e.get("wait_hrs", 0) > 0.001)
    print(f"\n  Queue complete. TC free at: {_fmt_h(tool_changer_free_at)}")
    print(f"  Events waited  : {n_waited} / {len(events)}")
    print(f"  Total wait fill: {round(total_wait_min, 1)} min  →  {total_extra_pcs:,.0f} extra pcs")


def stagger_changeovers(plan, machines, machine_hours):
    stagger_changeovers_serial_queue(plan, machines, machine_hours)

# =============================================================
# SECTION 21 — 22H UTILISATION ENFORCER  (FIX-17,18,19,21,22)
# =============================================================

def _add_part_to_machine(p, m, run_hrs, co_hrs, machine_hours,
                          machine_last_part, current_inventory,
                          already_planned, plan, priority_scores,
                          type_label, role_label):
    r_val   = rate.get(p, 1)
    qty     = round(run_hrs * r_val, 0)
    last    = machine_last_part.get(m)
    p_color = part_color.get(p,    "UNKNOWN")
    l_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
    has_purge = (
        last is not None and last != p
        and p_color != l_color
        and p_color != "UNKNOWN"
        and l_color not in ("UNKNOWN", "NEEDS_PURGE")
    )

    machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
    current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
    machine_last_part[m] = p
    already_planned.add(p)

    plan.append({
        "Part": p, "Color": p_color,
        "Category": part_category.get(p, "Stranger"),
        "Fixed_Machine": part_fixed_machine.get(p, "—"),
        "Fixed_Used": "N/A — enforcer", "Machine": m,
        "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
        "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty,
        "Monthly_Indent": round(indent_monthly.get(p, 0), 0),
        "Daily_Indent": round(indent_daily.get(p, 0), 2),
        "Today_Target": round(today_target_qty.get(p, 0), 0),
        "Changeover": "No" if co_hrs == 0 else "Yes",
        "Color_Purge": "Yes" if has_purge else "No",
        "Type": type_label, "Role": role_label,
        "Tools_Available": tools_available.get(p, 1),
        "Tools_Used": 1, "Runner_Lock": "No",
        "Priority_Score": round(priority_scores.get(p, 0), 2),
        "Phase": 1, "Stagger_Adjusted": "No",
    })
    return qty


def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id,
                          priority_scores):
    """
    V11 Utilisation Enforcer — Maximum Parts, Zero Idle.
    
    KEY CHANGES from V10:
    FIX-17: Never assign a part whose planned qty is met to a NEW machine.
            Only extend it on its EXISTING machine(s).
    FIX-18: Prioritise truly UNPLANNED parts (zero planned qty today)
            over extending parts that already have production somewhere.
    FIX-21: Parts with zero planned qty today always rank first in S2.
    FIX-22: Better candidate sorting — unplanned > below-safety > below-target > at-target.
    """
    print(f"\n  22H UTILIZATION ENFORCER  (floor={UTIL_TARGET_PCT}%  |  CO cap={MAX_DAILY_CO})")
    print(f"  FIX-17: Parts with met indent → extend only, no new machine")
    print(f"  FIX-18: Unplanned parts get absolute priority")
    micro_idle_log = []
    floor_hrs      = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)

    all_skipped = [
        p for p in all_parts
        if should_skip(p)[0]
        and rate.get(p, 0) > 0
        and indent_monthly.get(p, 0) > 0
        and part_fixed_machine.get(p) is None
    ]

    def _non_runner_already_on_machine():
        result = set()
        for row in plan:
            p = row["Part"]
            if part_category.get(p, "Stranger") in ("Stranger", "Repeater"):
                result.add(p)
        return result

    def _current_co_count():
        return sum(1 for r in plan if r.get("Changeover") == "Yes")

    # FIX-19: Track which parts have their indent already met
    def _parts_with_indent_met():
        """Returns set of parts whose total planned qty >= daily indent."""
        from collections import defaultdict
        part_qty = defaultdict(float)
        for row in plan:
            part_qty[row["Part"]] += float(row.get("Production_Qty", 0))
        result = set()
        for p, qty in part_qty.items():
            daily = indent_daily.get(p, 0)
            if daily > 0 and qty >= (daily - 0.5):
                result.add(p)
        return result

    machines_by_util = sorted(
        vt_machines, key=lambda m: machine_hours.get(m, 0)
    )

    for m in machines_by_util:
        if m in machine_fixed_parts:
            continue

        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue

        last_on_m  = machine_last_part.get(m)
        last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"

        # S0 — Extend existing parts to reach 90% floor
        current_util_hrs = machine_hours.get(m, 0)
        if current_util_hrs < floor_hrs:
            needed     = round(floor_hrs - current_util_hrs, 4)
            parts_on_m = [row for row in plan if row["Machine"] == m]
            parts_on_m_sorted = sorted(
                parts_on_m,
                key=lambda r: priority_scores.get(r["Part"], 0),
                reverse=True,
            )
            for row in parts_on_m_sorted:
                if needed <= 0.001 or remaining < 0.001:
                    break
                p_ext    = row["Part"]
                r_ext    = rate.get(p_ext, 1)
                daily_p  = indent_daily.get(p_ext, 0)
                inv_now  = current_inventory.get(p_ext, 0)
                headroom = max(0.0, opd_cap(scenario_id) * daily_p - inv_now)
                ext_hrs  = min(needed, remaining, headroom / r_ext if r_ext > 0 else 0)
                if ext_hrs < 0.001:
                    continue
                extra_qty = round(ext_hrs * r_ext, 0)
                row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                row["Total_Hrs_Used"] = round(
                    float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                row["Type"] = str(row.get("Type", "Primary")) + "+Floor90"
                machine_hours[m]         = round(machine_hours.get(m, 0) + ext_hrs, 4)
                current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
                remaining = round(remaining - ext_hrs, 4)
                needed    = round(needed    - ext_hrs, 4)
                print(f"    [S0-FLOOR90] {p_ext:28s} on {m:15s}  "
                      f"+{ext_hrs:.2f}h  qty+={extra_qty:.0f}  "
                      f"util={round(machine_hours.get(m,0)/AVAILABLE_HOURS*100,1)}%")

        if remaining < 0.05:
            continue

        # ─────────────────────────────────────────────────────
        # FIX-18: S2 BEFORE S1 — Add new UNPLANNED parts FIRST
        # This ensures we maximise the number of distinct parts
        # planned today before wasting time extending existing ones.
        # ─────────────────────────────────────────────────────
        if remaining >= MIN_RUN_HOURS:
            already_on_some_machine = _non_runner_already_on_machine()
            indent_met_parts = _parts_with_indent_met()

            unplanned = []
            for p in all_parts:
                # FIX-17: Skip parts whose indent is already met on another machine
                if p in indent_met_parts:
                    continue
                if p in already_planned and p not in indent_met_parts:
                    # Part is planned but indent NOT met — allow on this machine
                    # only if it's a Runner with spare tools
                    cat = part_category.get(p, "Stranger")
                    if cat != "Runner":
                        continue
                    machines_in_use = len({row["Machine"] for row in plan if row["Part"] == p})
                    if machines_in_use >= tools_available.get(p, 1):
                        continue
                elif p in already_planned:
                    continue
                if m not in vt_compat.get(p, []):
                    continue
                if should_skip(p)[0]:
                    continue
                if indent_monthly.get(p, 0) <= 0 or rate.get(p, 0) <= 0:
                    continue
                if current_inventory.get(p, 0) >= opd_cap(scenario_id) * indent_daily.get(p, 0):
                    continue
                if part_fixed_machine.get(p) is not None:
                    continue
                cat = part_category.get(p, "Stranger")
                if cat in ("Stranger", "Repeater") and p in already_on_some_machine:
                    continue
                unplanned.append(p)

            # FIX-21/22: Sort unplanned parts — truly unplanned first,
            # then by urgency
            def _enforcer_sort_key_v11(p):
                # Tier 0: Is this part completely unplanned today? (highest priority)
                total_qty_today = _get_part_total_qty(p, plan)
                is_unplanned = 0 if total_qty_today == 0 else 1
                
                # Tier 1: Same part / same color = no CO (prefer)
                needs_co = 0 if (last_on_m is None or last_on_m == p) else 1
                p_col    = part_color.get(p, "UNKNOWN")
                same_col = (
                    0 if (needs_co == 1 and p_col == last_color and p_col != "UNKNOWN")
                    else 1
                )
                
                # Tier 2: Category priority
                cat_pri  = {"Runner": 0, "Repeater": 1, "Stranger": 2}.get(
                    part_category.get(p, "Stranger"), 2)
                
                # Tier 3: Days coverage (lower = more urgent)
                inv_now  = current_inventory.get(p, 0)
                daily_p  = indent_daily.get(p, 0)
                days_cov = inv_now / daily_p if daily_p > 0 else 999
                
                # Tier 4: Priority score (higher = better, so negate)
                sc = priority_scores.get(p, 0)
                
                return (is_unplanned, needs_co, same_col, cat_pri, days_cov, -sc)

            unplanned.sort(key=_enforcer_sort_key_v11)

            for p in unplanned:
                if remaining < MIN_RUN_HOURS:
                    break
                co_hrs   = _co_hrs_for(p, m, machine_last_part)
                eff_free = round(remaining - co_hrs, 4)
                if eff_free < MIN_RUN_HOURS:
                    continue
                if co_hrs > 0 and _current_co_count() >= MAX_DAILY_CO:
                    continue

                daily_p  = indent_daily.get(p, 0)
                r_val    = rate.get(p, 1)
                inv_now  = current_inventory.get(p, 0)
                cap_qty  = opd_cap(scenario_id) * daily_p
                headroom = max(0.0, cap_qty - inv_now)
                if headroom <= 0:
                    continue

                shortfall = max(0.0, daily_p - inv_now)
                min_run  = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
                run_hrs  = max(min_run, min(eff_free, headroom / r_val if r_val > 0 else eff_free))

                cat = part_category.get(p, "Stranger")
                total_before = _get_part_total_qty(p, plan)
                was_unplanned = total_before == 0
                
                qty = _add_part_to_machine(
                    p, m, run_hrs, co_hrs,
                    machine_hours, machine_last_part, current_inventory,
                    already_planned, plan, priority_scores,
                    "Filler-Unplanned" if was_unplanned else "Filler-ShortfallExpand",
                    "Primary",
                )
                remaining = round(remaining - co_hrs - run_hrs, 4)
                co_tag = "No CO" if co_hrs == 0 else f"CO({part_color.get(p,'?')})"
                tag = "[NEW-PART]" if was_unplanned else "[SHORTFALL-EXPAND]"
                print(f"    [S2-UNPLAN]  {p:28s} ({cat}) → {m:15s}  "
                      f"{run_hrs:.2f}h  qty={qty:.0f}  [{co_tag}] {tag}")

        if remaining < 0.05:
            continue

        # S1 — NOW extend existing parts (after adding new ones)
        parts_on_machine = list({row["Part"] for row in plan if row["Machine"] == m})
        for p in sorted(parts_on_machine,
                        key=lambda x: priority_scores.get(x, 0), reverse=True):
            if remaining < 0.001:
                break
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            headroom = max(0.0, opd_cap(scenario_id) * daily_p - inv_now)
            if headroom <= 0:
                headroom = max(0.0, STRATEGIC_BUFFER_DAYS * daily_p - inv_now)
            ext_hrs  = min(remaining, headroom / r_val if r_val > 0 else 0)
            if ext_hrs < 0.001:
                continue
            extra_qty = round(ext_hrs * r_val, 0)
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "Primary")) + "+Extended"
                    break
            machine_hours[m]     = round(machine_hours.get(m, 0) + ext_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + extra_qty, 0)
            remaining = round(remaining - ext_hrs, 4)
            print(f"    [S1-EXTEND]  {p:28s} on {m:15s}  +{ext_hrs:.2f}h  qty+={extra_qty:.0f}")

        if remaining < MIN_RUN_HOURS:
            # Absorb tiny leftover
            if 0.001 < remaining:
                parts_on_m = [row for row in plan if row["Machine"] == m]
                parts_on_m_sorted = sorted(
                    parts_on_m,
                    key=lambda r: priority_scores.get(r["Part"], 0),
                    reverse=True,
                )
                for row in parts_on_m_sorted:
                    p_ext    = row["Part"]
                    if is_hard_skip(p_ext):
                        continue
                    r_ext    = rate.get(p_ext, 1)
                    daily_p  = indent_daily.get(p_ext, 0)
                    inv_now  = current_inventory.get(p_ext, 0)
                    strategic_cap = STRATEGIC_BUFFER_DAYS * daily_p
                    headroom = max(0.0, strategic_cap - inv_now)
                    ext_hrs  = min(remaining, headroom / r_ext if r_ext > 0 else 0)
                    if ext_hrs < 0.001:
                        continue
                    extra_qty = round(ext_hrs * r_ext, 0)
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "Primary")) + "+TailFill"
                    machine_hours[m]         = round(machine_hours.get(m, 0) + ext_hrs, 4)
                    current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
                    remaining = round(remaining - ext_hrs, 4)
                    print(f"    [S5-TAILFIL] {p_ext:28s} on {m:15s}  "
                          f"+{ext_hrs:.2f}h  qty+={extra_qty:.0f}  (tail fill)")
                    break
            continue

        # S3 — Re-run a RUNNER that has a spare tool
        # FIX-17: Only runners whose indent is NOT yet met
        indent_met_parts = _parts_with_indent_met()
        runner_spare = [
            p for p in already_planned
            if m in vt_compat.get(p, [])
            and part_category.get(p, "Stranger") == "Runner"
            and p not in indent_met_parts  # FIX-17: only if indent not met
            and m not in [row["Machine"] for row in plan if row["Part"] == p]
            and len({row["Machine"] for row in plan if row["Part"] == p})
                < tools_available.get(p, 1)
            and part_fixed_machine.get(p) is None
        ]
        runner_spare.sort(key=lambda p: (
            0 if (last_on_m is None or last_on_m == p) else 1,
            0 if (part_color.get(p, "UNKNOWN") == last_color and last_color != "UNKNOWN") else 1,
            -priority_scores.get(p, 0),
        ))

        for p in runner_spare:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            if co_hrs > 0 and _current_co_count() >= MAX_DAILY_CO:
                continue

            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            headroom = max(0.0, opd_cap(scenario_id) * daily_p - inv_now)
            if headroom <= 0:
                continue

            shortfall = max(0.0, daily_p - inv_now)
            min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
            run_hrs   = max(min_run, min(eff_free, headroom / r_val if r_val > 0 else eff_free))

            qty = round(run_hrs * r_val, 0)
            machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
            machine_last_part[m] = p
            remaining            = round(remaining - co_hrs - run_hrs, 4)

            p_color        = part_color.get(p, "UNKNOWN")
            has_purge_s3   = co_hrs > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001
            tools_used_now = len({row["Machine"] for row in plan if row["Part"] == p}) + 1

            plan.append({
                "Part": p, "Color": p_color, "Category": "Runner",
                "Fixed_Machine": part_fixed_machine.get(p, "—"),
                "Fixed_Used": "N/A — enforcer rerun", "Machine": m,
                "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
                "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
                "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty,
                "Monthly_Indent": round(indent_monthly.get(p, 0), 0),
                "Daily_Indent": round(daily_p, 2),
                "Today_Target": round(today_target_qty.get(p, 0), 0),
                "Changeover": "No" if co_hrs == 0 else "Yes",
                "Color_Purge": "Yes" if has_purge_s3 else "No",
                "Type": "Runner-Rerun (spare tool, indent not met)",
                "Role": f"Tool-Expansion (tool {tools_used_now})",
                "Tools_Available": tools_available.get(p, 1),
                "Tools_Used": tools_used_now, "Runner_Lock": "No",
                "Priority_Score": round(priority_scores.get(p, 0), 2),
                "Phase": 3, "Stagger_Adjusted": "No",
            })
            print(f"    [S3-RUNNER]  {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  tool {tools_used_now}/{tools_available.get(p,1)}")
            break

        if remaining < MIN_RUN_HOURS:
            continue

        # S4 — Skipped parts as last resort
        already_on_some_machine = _non_runner_already_on_machine()

        skipped_candidates = []
        for p in all_skipped:
            if m not in vt_compat.get(p, []):
                continue
            if terminal_blocked(p)[0]:
                continue
            cat = part_category.get(p, "Stranger")
            if cat in ("Stranger", "Repeater") and p in already_on_some_machine:
                continue
            # FIX-17: Don't plan a skipped part on a NEW machine if
            # it's already planned and indent met elsewhere
            if p in _parts_with_indent_met():
                continue
            skipped_candidates.append(p)

        skipped_candidates.sort(key=lambda p: (
            0 if (last_on_m is None or last_on_m == p) else 1,
            0 if (part_color.get(p, "UNKNOWN") == last_color and last_color != "UNKNOWN") else 1,
            {"Runner": 0, "Repeater": 1, "Stranger": 2}.get(part_category.get(p, "Stranger"), 2),
            -indent_daily.get(p, 0),
        ))

        for p in skipped_candidates:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            if co_hrs > 0 and _current_co_count() >= MAX_DAILY_CO:
                continue

            r_val   = rate.get(p, 1)
            run_hrs = max(
                MIN_RUN_HOURS,
                min(eff_free, indent_monthly.get(p, 0) / r_val if r_val > 0 else eff_free),
            )
            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Skipped (last resort)", "Primary",
            )
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S4-SKIPPED] {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  cat={part_category.get(p,'?')}")
            break

        # S5 — Absorb any tiny leftover
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if 0.001 < remaining < MIN_RUN_HOURS:
            parts_on_m = [row for row in plan if row["Machine"] == m]
            parts_on_m_sorted = sorted(
                parts_on_m,
                key=lambda r: priority_scores.get(r["Part"], 0),
                reverse=True,
            )
            for row in parts_on_m_sorted:
                p_ext    = row["Part"]
                if is_hard_skip(p_ext):
                    continue
                r_ext    = rate.get(p_ext, 1)
                daily_p  = indent_daily.get(p_ext, 0)
                inv_now  = current_inventory.get(p_ext, 0)
                strategic_cap = STRATEGIC_BUFFER_DAYS * daily_p
                headroom = max(0.0, strategic_cap - inv_now)
                ext_hrs  = min(remaining, headroom / r_ext if r_ext > 0 else 0)
                if ext_hrs < 0.001:
                    continue
                extra_qty = round(ext_hrs * r_ext, 0)
                row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                row["Total_Hrs_Used"] = round(
                    float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                row["Type"] = str(row.get("Type", "Primary")) + "+TailFill"
                machine_hours[m]         = round(machine_hours.get(m, 0) + ext_hrs, 4)
                current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
                remaining = round(remaining - ext_hrs, 4)
                print(f"    [S5-TAILFIL] {p_ext:28s} on {m:15s}  "
                      f"+{ext_hrs:.2f}h  qty+={extra_qty:.0f}  (tail fill)")
                break

        final_remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_remaining >= 0.25:
            util_final = round((1 - final_remaining / AVAILABLE_HOURS) * 100, 1)
            micro_idle_log.append({
                "Machine":         m,
                "Idle_Hrs":        round(final_remaining, 3),
                "Utilization_Pct": util_final,
                "Note":            "Exhausted all compatible parts at OPD ceiling",
            })
            print(f"    [⚠ IDLE]     {m:15s}  "
                  f"{final_remaining:.2f}h idle ({util_final}%)")

    return micro_idle_log

# =============================================================
# SECTION 21B — POST-PLAN VALIDATION
# =============================================================

def validate_plan_rows(plan, current_inventory):
    violations = []
    for row in plan:
        p         = row["Part"]
        run_h     = float(row.get("Run_Hours", 0) or 0)
        qty       = float(row.get("Production_Qty", 0) or 0)
        daily     = indent_daily.get(p, 0)

        v = []
        if run_h < MIN_RUN_HOURS - 0.001:
            v.append(f"Run_Hours={run_h:.3f} < MIN={MIN_RUN_HOURS}")
        if daily > 0 and qty < daily - 0.5:
            v.append(f"Qty={qty:.0f} < Daily_Indent={daily:.2f}")

        if v:
            row["Indent_Met"] = "VIOLATION: " + " | ".join(v)
            violations.append({
                "Part": p, "Machine": row.get("Machine", "—"),
                "Run_Hours": run_h, "Qty": qty,
                "Daily_Indent": daily, "Violations": " | ".join(v),
            })

    if violations:
        print(f"\n  ⚠ POST-PLAN VALIDATION: {len(violations)} row(s) violate hard rules")
        for v in violations:
            print(f"    {v['Part']:<28} {v['Machine']:<15} {v['Violations']}")
    else:
        print(f"\n  Post-plan validation: all rows OK  ✓")

    return violations

# =============================================================
# SECTION 21B2 — STRATEGIC BUFFER FILLER PASS (FIX-17 integrated)
# =============================================================

def strategic_buffer_score(part, current_inventory):
    daily    = indent_daily.get(part, 0.0)
    inv      = current_inventory.get(part, 0.0)
    r_val    = rate.get(part, 1.0)
    cat      = part_category.get(part, "Stranger")

    if daily <= 0 or r_val <= 0:
        return 0.0

    velocity = daily / max(float(inv), 1.0)
    cat_w = {"Runner": 1.0, "Repeater": 0.7, "Stranger": 0.4}.get(cat, 0.4)
    return round(velocity * cat_w * STRATEGIC_PRIORITY_DISCOUNT * 100, 2)


def strategic_buffer_filler(plan, machine_hours, machine_last_part,
                              all_parts, already_planned,
                              current_inventory, scenario_id):
    """
    V11 Strategic Buffer Filler — with FIX-17 integrated.
    
    KEY RULE: A part whose daily indent is already met across all
    machines CANNOT be assigned to a NEW machine. It can only be
    EXTENDED on machines where it already runs.
    
    This prevents the old bug where a part with met indent was
    "planned" on idle machines just to fill time, wasting a
    changeover and tool slot that could serve an unplanned part.
    """
    print(f"\n{'─'*65}")
    print(f"  STRATEGIC BUFFER FILLER PASS (V11 — FIX-17 active)")
    print(f"  Rule: indent-met parts → extend only, no new machine")
    print(f"  Ceiling: {STRATEGIC_BUFFER_DAYS} days  |  Util floor: {EFFICIENCY_MODE_UTIL_FLOOR}%")
    print(f"{'─'*65}")

    filled_count = 0
    ABSOLUTE_MAX_DAYS = 2 * TARGET_DAYS

    def _current_co_count():
        return sum(1 for r in plan if r.get("Changeover") == "Yes")

    def _non_runner_on_machine():
        result = set()
        for row in plan:
            p = row["Part"]
            if part_category.get(p, "Stranger") in ("Stranger", "Repeater"):
                result.add(p)
        return result

    def _parts_with_indent_met():
        from collections import defaultdict
        part_qty = defaultdict(float)
        for row in plan:
            part_qty[row["Part"]] += float(row.get("Production_Qty", 0))
        result = set()
        for p, qty in part_qty.items():
            daily = indent_daily.get(p, 0)
            if daily > 0 and qty >= (daily - 0.5):
                result.add(p)
        return result

    machines_by_util = sorted(
        vt_machines, key=lambda m: machine_hours.get(m, 0)
    )

    for m in machines_by_util:
        if m in machine_fixed_parts:
            continue

        used_hrs  = machine_hours.get(m, 0)
        remaining = round(AVAILABLE_HOURS - used_hrs, 4)

        if remaining < 0.001:
            continue

        util_pct = round(used_hrs / AVAILABLE_HOURS * 100, 1)
        last_on_m  = machine_last_part.get(m)
        last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"
        co_cap_hit = _current_co_count() >= MAX_DAILY_CO

        print(f"\n  Machine {m:20s}  used={used_hrs:.2f}h  free={remaining:.2f}h  "
              f"util={util_pct:.1f}%  CO_cap={'HIT' if co_cap_hit else 'OK'}")

        # Step A: Extend existing parts on this machine (no CO cost)
        parts_on_m = sorted(
            [row for row in plan if row["Machine"] == m],
            key=lambda r: strategic_buffer_score(r["Part"], current_inventory),
            reverse=True,
        )
        for row in parts_on_m:
            if remaining < 0.001:
                break
            p_ext    = row["Part"]
            if is_hard_skip(p_ext):
                continue
            r_ext    = rate.get(p_ext, 1)
            daily_p  = indent_daily.get(p_ext, 0)
            inv_now  = current_inventory.get(p_ext, 0)
            strategic_cap = STRATEGIC_BUFFER_DAYS * daily_p
            headroom = max(0.0, strategic_cap - inv_now)
            ext_hrs  = min(remaining, headroom / r_ext if r_ext > 0 else 0)
            if ext_hrs < 0.001:
                continue
            extra_qty = round(ext_hrs * r_ext, 0)
            row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
            row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
            row["Total_Hrs_Used"] = round(
                float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
            row["Type"] = str(row.get("Type", "")) + "+StrategicBuffer"
            machine_hours[m]         = round(machine_hours.get(m, 0) + ext_hrs, 4)
            current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
            remaining = round(remaining - ext_hrs, 4)
            filled_count += 1
            new_days = current_inventory.get(p_ext, 0) / daily_p if daily_p > 0 else 0
            print(f"    [SB-EXTEND]  {p_ext:28s}  +{ext_hrs:.2f}h  "
                  f"qty+={extra_qty:.0f}  days_cov→{new_days:.2f}")

        if remaining < MIN_RUN_HOURS:
            # Absorb tiny remainder
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            if remaining > 0.001:
                for row in parts_on_m:
                    if remaining < 0.001:
                        break
                    p_ext = row["Part"]
                    if is_hard_skip(p_ext):
                        continue
                    r_ext = rate.get(p_ext, 1)
                    daily_p = indent_daily.get(p_ext, 0)
                    inv_now_e = current_inventory.get(p_ext, 0)
                    headroom = max(0.0, ABSOLUTE_MAX_DAYS * daily_p - inv_now_e)
                    ext_hrs = min(remaining, headroom / r_ext if r_ext > 0 else 0)
                    if ext_hrs < 0.001:
                        continue
                    extra_qty = round(ext_hrs * r_ext, 0)
                    row["Run_Hours"] = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "")) + "+MaxBuffer"
                    machine_hours[m] = round(machine_hours.get(m, 0) + ext_hrs, 4)
                    current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
                    remaining = round(remaining - ext_hrs, 4)
                    filled_count += 1
            continue

        # Step B: Add NEW parts (only if CO cap not hit)
        # FIX-17: NEVER add a part whose indent is already met on another machine
        if not co_cap_hit:
            already_on_some = _non_runner_on_machine()
            indent_met_parts = _parts_with_indent_met()

            candidates = []
            for p in all_parts:
                if m not in vt_compat.get(p, []):
                    continue
                if is_hard_skip(p):
                    continue
                if part_fixed_machine.get(p) is not None:
                    continue
                # FIX-17: CRITICAL — skip parts whose indent is met
                if p in indent_met_parts:
                    continue
                cat = part_category.get(p, "Stranger")
                if cat in ("Stranger", "Repeater") and p in already_on_some:
                    continue
                if cat == "Runner" and p in already_planned:
                    machines_in_use = len({row["Machine"] for row in plan if row["Part"] == p})
                    if machines_in_use >= tools_available.get(p, 1):
                        continue
                inv_p   = current_inventory.get(p, 0)
                daily_p = indent_daily.get(p, 0)
                if daily_p > 0 and inv_p >= STRATEGIC_BUFFER_DAYS * daily_p:
                    continue
                candidates.append(p)

            def _strat_sort(p):
                # Truly unplanned parts first
                total_qty_today = _get_part_total_qty(p, plan)
                is_unplanned = 0 if total_qty_today == 0 else 1
                needs_co = 0 if (last_on_m is None or last_on_m == p) else 1
                p_col    = part_color.get(p, "UNKNOWN")
                same_col = (
                    0 if (needs_co == 1 and p_col == last_color and p_col != "UNKNOWN")
                    else 1
                )
                sc = strategic_buffer_score(p, current_inventory)
                return (is_unplanned, needs_co, same_col, -sc)

            candidates.sort(key=_strat_sort)

            for p in candidates:
                if remaining < MIN_RUN_HOURS:
                    break
                co_hrs   = _co_hrs_for(p, m, machine_last_part)
                eff_free = round(remaining - co_hrs, 4)
                if eff_free < MIN_RUN_HOURS:
                    continue
                if co_hrs > 0 and _current_co_count() >= MAX_DAILY_CO:
                    continue

                r_val    = rate.get(p, 1)
                daily_p  = indent_daily.get(p, 0)
                inv_now  = current_inventory.get(p, 0)
                strategic_cap = STRATEGIC_BUFFER_DAYS * daily_p
                headroom = max(0.0, strategic_cap - inv_now)
                if headroom <= 0:
                    continue

                run_hrs  = max(MIN_RUN_HOURS, min(eff_free, headroom / r_val if r_val > 0 else eff_free))
                qty      = round(run_hrs * r_val, 0)
                p_color  = part_color.get(p, "UNKNOWN")
                has_purge = co_hrs > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001

                machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
                current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
                machine_last_part[m] = p
                already_planned.add(p)
                remaining = round(remaining - co_hrs - run_hrs, 4)
                filled_count += 1

                inv_after = current_inventory.get(p, 0)
                new_days  = inv_after / daily_p if daily_p > 0 else 0
                co_tag    = "No CO" if co_hrs == 0 else f"CO({p_color})"
                cat_tag   = part_category.get(p, "?")
                was_unplanned = _get_part_total_qty(p, plan) == qty  # only this row

                plan.append({
                    "Part": p, "Color": p_color, "Category": cat_tag,
                    "Fixed_Machine": part_fixed_machine.get(p, "—"),
                    "Fixed_Used": "N/A — strategic buffer", "Machine": m,
                    "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
                    "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty,
                    "Monthly_Indent": round(indent_monthly.get(p, 0), 0),
                    "Daily_Indent": round(daily_p, 2), "Today_Target": 0,
                    "Changeover": "No" if co_hrs == 0 else "Yes",
                    "Color_Purge": "Yes" if has_purge else "No",
                    "Type": "Strategic-Buffer" + (" [NEW-PART]" if was_unplanned else ""),
                    "Role": "Buffer-Fill", "Tools_Available": tools_available.get(p, 1),
                    "Tools_Used": 1, "Runner_Lock": "No",
                    "Priority_Score": round(strategic_buffer_score(p, current_inventory), 2),
                    "Phase": 4, "Stagger_Adjusted": "No",
                })
                print(f"    [SB-NEW]     {p:28s} ({cat_tag}) → {m:15s}  "
                      f"{run_hrs:.2f}h  qty={qty:.0f}  [{co_tag}]  "
                      f"days_cov→{new_days:.2f}"
                      f"{'  [NEW-PART ✓]' if was_unplanned else ''}")

        # Step C: Unconditional extend to eliminate all idle
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining > 0.001:
            parts_on_m_now = sorted(
                [row for row in plan if row["Machine"] == m],
                key=lambda r: strategic_buffer_score(r["Part"], current_inventory),
                reverse=True,
            )

            # Pass 1: strategic ceiling
            for row in parts_on_m_now:
                if remaining < 0.001:
                    break
                p_ext     = row["Part"]
                if is_hard_skip(p_ext):
                    continue
                r_ext     = rate.get(p_ext, 1)
                daily_p   = indent_daily.get(p_ext, 0)
                inv_now_e = current_inventory.get(p_ext, 0)
                headroom  = max(0.0, STRATEGIC_BUFFER_DAYS * daily_p - inv_now_e)
                ext_hrs   = min(remaining, headroom / r_ext if r_ext > 0 else 0)
                if ext_hrs < 0.001:
                    continue
                extra_qty = round(ext_hrs * r_ext, 0)
                row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                row["Total_Hrs_Used"] = round(
                    float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                row["Type"] = str(row.get("Type", "")) + "+CO-Cap-Extend"
                machine_hours[m]         = round(machine_hours.get(m, 0) + ext_hrs, 4)
                current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
                remaining = round(remaining - ext_hrs, 4)
                filled_count += 1

            # Pass 2: absolute max
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            if remaining > 0.001:
                for row in parts_on_m_now:
                    if remaining < 0.001:
                        break
                    p_ext     = row["Part"]
                    if is_hard_skip(p_ext):
                        continue
                    r_ext     = rate.get(p_ext, 1)
                    daily_p   = indent_daily.get(p_ext, 0)
                    inv_now_e = current_inventory.get(p_ext, 0)
                    headroom  = max(0.0, ABSOLUTE_MAX_DAYS * daily_p - inv_now_e)
                    ext_hrs   = min(remaining, headroom / r_ext if r_ext > 0 else 0)
                    if ext_hrs < 0.001:
                        continue
                    extra_qty = round(ext_hrs * r_ext, 0)
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "")) + "+MaxBuffer"
                    machine_hours[m]         = round(machine_hours.get(m, 0) + ext_hrs, 4)
                    current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
                    remaining = round(remaining - ext_hrs, 4)
                    filled_count += 1

        final_used = machine_hours.get(m, 0)
        final_util = round(final_used / AVAILABLE_HOURS * 100, 1)
        final_idle = round(AVAILABLE_HOURS - final_used, 4)
        if final_idle > 0.05:
            print(f"    → {m:20s}  final util={final_util:.1f}%  "
                  f"IDLE={final_idle:.2f}h  (all parts at absolute ceiling or blocked)")
        else:
            status = "FULL ✓" if final_idle < 0.01 else f"≈FULL ({final_idle:.2f}h)"
            print(f"    → {m:20s}  final util={final_util:.1f}%  {status}")

    print(f"\n  Strategic buffer pass complete: {filled_count} extension(s)/addition(s)")
    return filled_count

# =============================================================
# SECTION 21C — FORWARD LOOK
# =============================================================

def compute_forward_look(current_inventory_after, all_parts, scenario_id):
    rows = []
    for p in all_parts:
        daily   = indent_daily.get(p, 0)
        monthly = indent_monthly.get(p, 0)
        if daily <= 0 or monthly <= 0:
            continue
        inv_now = current_inventory_after.get(p, 0)
        days_now = inv_now / daily if daily > 0 else 999

        days_until_safety = max(0.0, round((inv_now - SAFETY_DAYS * daily) / daily, 1))
        days_until_zero   = max(0.0, round(inv_now / daily, 1))

        alert = ""
        if days_until_zero <= FORWARD_LOOK_DAYS:
            alert = f"ZERO-STOCK RISK in {days_until_zero:.1f} days"
        elif days_until_safety <= FORWARD_LOOK_DAYS:
            alert = f"BELOW SAFETY in {days_until_safety:.1f} days"

        if alert:
            rows.append({
                "Part": p, "Color": part_color.get(p, "UNKNOWN"),
                "Category": part_category.get(p, "Stranger"),
                "Fixed_Machine": part_fixed_machine.get(p, "—"),
                "Daily_Indent": round(daily, 2),
                "Inv_After_Today": round(inv_now, 0),
                "Days_Coverage_Today": round(days_now, 2),
                "Days_Until_Safety": days_until_safety,
                "Days_Until_Zero": days_until_zero,
                "Alert": alert,
                "Action": (
                    "ESCALATE — schedule tomorrow without fail"
                    if days_until_zero <= 2
                    else "Plan production in next 1–3 days"
                ),
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("Days_Until_Zero").reset_index(drop=True)
    return df

# =============================================================
# SECTION 22 — OUTPUT VIEW BUILDERS
# =============================================================

def build_multi_machine_view(plan):
    if not plan:
        return pd.DataFrame()
    from collections import defaultdict
    part_rows = defaultdict(list)
    for row in plan:
        part_rows[row["Part"]].append(row)
    multi = {p: rows for p, rows in part_rows.items() if len(rows) > 1}
    if not multi:
        return pd.DataFrame()

    output_rows = []
    for part, rows in sorted(multi.items(), key=lambda x: -sum(r["Production_Qty"] for r in x[1])):
        daily     = indent_daily.get(part, 0)
        total_qty = sum(float(r["Production_Qty"]) for r in rows)
        for row in rows:
            output_rows.append({
                "Part": part, "Color": part_color.get(part, "UNKNOWN"),
                "Category": part_category.get(part, "Stranger"),
                "Fixed_Machine": part_fixed_machine.get(part, "—"),
                "Tools_Available": tools_available.get(part, 1),
                "Machines_Used": len(rows), "Machine": row["Machine"],
                "Role": row.get("Role", "Primary"),
                "Run_Hours": round(float(row["Run_Hours"]), 2),
                "Changeover_Hrs": round(float(row.get("Changeover_Hrs", 0)), 2),
                "Color_Purge": row.get("Color_Purge", "No"),
                "Production_Qty": round(float(row["Production_Qty"]), 0),
                "Daily_Indent": round(daily, 2),
                "Total_Qty_All_Machines": round(total_qty, 0),
                "Type": row.get("Type", "—"),
            })
        output_rows.append({
            "Part": f"  ↳ TOTAL — {part}", "Color": part_color.get(part, "UNKNOWN"),
            "Category": "—", "Fixed_Machine": part_fixed_machine.get(part, "—"),
            "Tools_Available": tools_available.get(part, 1),
            "Machines_Used": len(rows), "Machine": f"{len(rows)} machines",
            "Role": "TOTAL",
            "Run_Hours": round(sum(float(r["Run_Hours"]) for r in rows), 2),
            "Changeover_Hrs": round(sum(float(r.get("Changeover_Hrs",0)) for r in rows), 2),
            "Color_Purge": "—", "Production_Qty": round(total_qty, 0),
            "Daily_Indent": round(daily, 2),
            "Total_Qty_All_Machines": round(total_qty, 0), "Type": "—",
        })
        output_rows.append({k: "" for k in output_rows[-1].keys()})
    return pd.DataFrame(output_rows)


def build_production_vs_indent(plan, all_parts):
    if not plan:
        return pd.DataFrame()
    from collections import defaultdict
    part_qty      = defaultdict(float)
    part_machines = defaultdict(list)
    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])

    rows = []
    for p in sorted(part_qty.keys()):
        daily     = indent_daily.get(p, 0)
        monthly   = indent_monthly.get(p, 0)
        inv_b     = inventory.get(p, 0)
        produced  = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap       = round(produced - daily, 0)
        gap_dir   = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")
        extra_days = round(gap / daily, 2) if daily > 0 and gap > 0 else 0.0
        rows.append({
            "Part": p, "Color": part_color.get(p, "UNKNOWN"),
            "Category": part_category.get(p, "Stranger"),
            "Fixed_Machine": part_fixed_machine.get(p, "—"),
            "Tools_Available": tools_available.get(p, 1),
            "Machines": ", ".join(dict.fromkeys(part_machines[p])),
            "Machines_Count": len(set(part_machines[p])),
            "Total_Qty_Produced": produced, "Daily_Indent": round(daily, 2),
            "Monthly_Indent": round(monthly, 0), "Gap_vs_Daily": gap,
            "Gap_Direction": gap_dir, "Extra_Days_Stock": extra_days,
            "Inventory_Before": round(inv_b, 0), "Inventory_After": inv_after,
            "Days_Coverage_After": round(inv_after / daily, 2) if daily > 0 else 0,
        })

    order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = df.sort_values(["_sort", "Gap_vs_Daily"]).drop(columns=["_sort"]).reset_index(drop=True)
    return df


def build_inventory_target_sheet(plan, all_parts, scenario_id):
    from collections import defaultdict
    part_produced = defaultdict(float)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))

    rows = []
    for p in sorted(all_parts):
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_produced.get(p, 0), 0)
        inv_after   = round(inv_b + produced, 0)
        days_before = round(inv_b     / daily, 2) if daily > 0 else 0
        days_after  = round(inv_after / daily, 2) if daily > 0 else 0
        target_qty  = round(TARGET_DAYS * daily, 0)
        gap_qty     = round(target_qty - inv_after, 0)
        gap_days    = max(0, round(gap_qty / daily, 2) if daily > 0 else 0)
        cap         = opd_cap(scenario_id)
        max_prod    = round(cap * daily, 0)
        r_val       = rate.get(p, 1)
        indent_hrs  = daily / r_val if r_val > 0 else 0

        if inv_after == 0:
            status = "CRITICAL"
        elif days_after < SAFETY_DAYS:
            status = "BELOW_SAFETY"
        elif days_after < TARGET_DAYS:
            status = "BUILDING"
        else:
            status = "AT_TARGET"

        net_gain = (cap - 1) * daily if daily > 0 else 0
        if gap_qty <= 0:
            est_days = "AT TARGET"
        elif net_gain <= 0:
            est_days = "N/A"
        else:
            est_days = str(math.ceil(gap_qty / net_gain)) + " days"

        skip, skip_reason = should_skip(p)
        fm = part_fixed_machine.get(p, "—")
        rows.append({
            "Part": p, "Color": part_color.get(p, "UNKNOWN"),
            "Category": part_category.get(p, "Stranger"),
            "Fixed_Machine": fm,
            "Fixed_Phase": (
                "A — locked" if fm != "—" and fm in _phase_a_machines else
                "B — free"   if fm != "—" else "—"
            ),
            "Tools": tools_available.get(p, 1),
            "Monthly_Indent": round(monthly, 0), "Daily_Indent": round(daily, 2),
            "Indent_Hrs_Daily": round(indent_hrs, 2),
            "Target_Qty_5days": target_qty,
            "Safety_Floor_Qty_3days": round(SAFETY_DAYS * daily, 0),
            "Inv_Before": round(inv_b, 0), "Days_Coverage_Before": days_before,
            "Produced_Today": produced, "Inv_After": inv_after,
            "Days_Coverage_After": days_after,
            "Gap_to_Target_Qty": max(0, gap_qty),
            "Gap_to_Target_Days": gap_days, "Buffer_Status": status,
            "OPD_Cap_Today_Days": cap, "Max_Producible_Qty": max_prod,
            "Est_Days_to_Target": est_days,
            "Scheduled_Today": (
                "YES" if produced > 0
                else "SKIPPED — AT TARGET" if inv_b >= target_qty
                else "NO — no capacity"
            ),
            "Skip_Reason": skip_reason if skip else "",
        })

    status_order = {"CRITICAL": 0, "BELOW_SAFETY": 1, "BUILDING": 2, "AT_TARGET": 3}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Buffer_Status"].map(status_order)
        df = df.sort_values(["_sort", "Gap_to_Target_Days"], ascending=[True, False]) \
               .drop(columns=["_sort"]).reset_index(drop=True)
    return df


def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv     = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily   = indent_daily.get(p, 0.0)
        target  = today_target_qty.get(p, 0.0)
        r       = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0
        days_cov   = inv / daily if daily > 0 else 0
        fm = part_fixed_machine.get(p, "—")

        if inv == 0.0 and monthly > 0:
            status = "ZERO INV — FORCED"
        elif skip and inv >= TARGET_DAYS * daily:
            status = "AT 5-DAY TARGET — SKIP"
        elif skip:
            status = "SKIPPED"
        elif target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"

        rows.append({
            "Part": p, "Color": part_color.get(p, "UNKNOWN"),
            "Fixed_Machine": fm, "Tools": tools_available.get(p, 1),
            "Monthly_Indent": round(monthly, 0),
            "Indent_Hrs_Total": round(indent_hrs, 2),
            "Daily_Indent": round(daily, 2),
            "Indent_Hrs_Daily": round(daily / r if r > 0 else 0, 2),
            "Working_Days": WORKING_DAYS, "Inventory_Now": round(inv, 0),
            "Days_Coverage": round(days_cov, 2),
            "Safety_Floor": SAFETY_DAYS, "Target_Ceiling": TARGET_DAYS,
            "Today_Target_Qty": round(target, 0),
            "Today_Target_Hrs": round(target / r if r > 0 else 0, 2),
            "Rate_Per_Hour": round(r, 2), "Indent_Status": status,
            "Skip_Reason": skip_reason,
        })
    return pd.DataFrame(rows)


def build_terminal_status_sheet():
    rows = []
    all_terminals_known = set(terminal_status.keys())
    for terms in part_terminals.values():
        for t in terms:
            all_terminals_known.add(t)

    for t in sorted(all_terminals_known):
        inv           = terminal_status.get(t, None)
        parts_needing = [p for p, terms in part_terminals.items() if t in terms]
        parts_blocked = []
        for p in parts_needing:
            cat    = part_category.get(p, "Stranger")
            daily  = indent_daily.get(p, 0)
            mult   = TERMINAL_THRESHOLD.get(cat, 0.25)
            thresh = mult * daily
            if inv is None or inv < thresh:
                parts_blocked.append(f"{p}(need≥{thresh:.0f})")

        rows.append({
            "Terminal": t,
            "Inventory": round(inv, 0) if inv is not None else "NOT IN SHEET",
            "Parts_Requiring_Count": len(parts_needing),
            "Parts_Requiring": ", ".join(sorted(parts_needing)) if parts_needing else "—",
            "Parts_Blocked_Count": len(parts_blocked),
            "Parts_Blocked": ", ".join(parts_blocked) if parts_blocked else "—",
            "Threshold_Rule": (
                "Runner≥1×daily | Repeater≥0.5×daily | Stranger≥0.25×daily"
                if parts_needing else "—"
            ),
            "Impact": (
                "BLOCKING — reschedule" if parts_blocked
                else "Adequate today"   if parts_needing
                else "No parts"
            ),
        })
    return pd.DataFrame(rows)


def build_fixed_machine_status(plan, current_inventory):
    rows = []
    for machine, fixed_parts in sorted(machine_fixed_parts.items()):
        phase = "A" if machine in _phase_a_machines else "B"
        parts_on_machine_today = [r for r in plan if r["Machine"] == machine]
        part_run_today = (
            parts_on_machine_today[0]["Part"] if parts_on_machine_today else "—"
        )
        hrs_used = sum(
            float(r.get("Run_Hours", 0)) + float(r.get("Changeover_Hrs", 0))
            for r in parts_on_machine_today
        )

        for p in fixed_parts:
            daily    = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_b    = inventory.get(p, 0)
            inv_now  = current_inventory.get(p, inv_b)
            days_b   = round(inv_b   / daily, 2) if daily > 0 else 0
            days_now = round(inv_now / daily, 2) if daily > 0 else 0
            produced = round(inv_now - inv_b, 0)
            target_inv    = SAFETY_DAYS * daily
            gap_to_buffer = max(0, round(target_inv - inv_now, 0))
            indent_hrs    = daily / r_val if r_val > 0 else 0
            cap_hrs = AVAILABLE_HOURS_EXTENDED if indent_hrs > 20 else AVAILABLE_HOURS
            extra_hrs_per_day = max(0.0, cap_hrs - indent_hrs)
            extra_qty_per_day = extra_hrs_per_day * r_val if r_val > 0 else 0
            days_to_buffer = (
                math.ceil(gap_to_buffer / extra_qty_per_day)
                if extra_qty_per_day > 0 and gap_to_buffer > 0
                else ("AT BUFFER" if gap_to_buffer <= 0 else "N/A")
            )

            rows.append({
                "Machine": machine, "Phase": f"Phase {phase}",
                "Part": p, "Part_Run_Today": part_run_today,
                "Ran_Today": "YES" if part_run_today == p else "NO",
                "Category": part_category.get(p, "Runner"),
                "Color": part_color.get(p, "UNKNOWN"),
                "Daily_Indent": round(daily, 2),
                "Indent_Hrs_Daily": round(indent_hrs, 2),
                "Machine_Cap_Hrs": cap_hrs if phase == "A" else AVAILABLE_HOURS,
                "Inv_Start_of_Day": round(inv_b, 0),
                "Days_Coverage_Before": days_b,
                "Produced_Today": produced,
                "Inv_End_of_Day": round(inv_now, 0),
                "Days_Coverage_After": days_now,
                "Buffer_Target_Qty": round(target_inv, 0),
                "Gap_to_3day_Buffer": round(gap_to_buffer, 0),
                "Est_Days_to_Buffer": str(days_to_buffer),
                "Machine_Hrs_Used": round(hrs_used, 2),
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["Machine", "Part"]).reset_index(drop=True)
    return df


SPECIALIZED_MACHINE_THRESHOLD = 3

def detect_specialized_machines(all_parts):
    machine_parts = {}
    for m in vt_machines:
        machine_parts[m] = [
            p for p in all_parts
            if m in vt_compat.get(p, [])
            and not should_skip(p)[0]
            and indent_monthly.get(p, 0) > 0
        ]
    specialized = {
        m for m, mp in machine_parts.items()
        if 0 < len(mp) <= SPECIALIZED_MACHINE_THRESHOLD
    }
    return (
        specialized,
        {m: machine_parts[m] for m in specialized},
        machine_parts,
    )

# =============================================================
# SECTION 23 — MAIN SCHEDULER
# =============================================================

def schedule(parts, label=""):
    global _phase_a_machines

    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(vt_machines)} machines")
    print(f"  Safety floor: {SAFETY_DAYS} days  |  Target ceiling: {TARGET_DAYS} days")
    print(f"{'─'*65}")

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")
    print(f"  Effective OPD cap today: {opd_cap(scenario_id)} days")

    horizon_df = compute_indent_horizon(parts)

    active_parts = [
        p for p in parts
        if not should_skip(p)[0] and indent_monthly.get(p, 0) > 0
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    machine_hours     = {m: 0.0 for m in vt_machines}
    machine_last_part = {m: machine_state.get(m) for m in vt_machines}
    current_inventory = inventory.copy()

    inventory_start_of_day = inventory.copy()

    plan          = []
    already_planned = set()
    not_planned   = []
    deferred      = []

    # FIXED MACHINE PASS
    _phase_a_machines, fixed_plan_rows = schedule_fixed_machines(
        machine_hours, machine_last_part,
        current_inventory, plan, already_planned,
        priority_scores, scenario_id,
    )

    # Specialized machines
    specialized_machines, spec_machine_parts, _ = detect_specialized_machines(list(parts))
    print(f"\n  SPECIALIZED MACHINES (≤{SPECIALIZED_MACHINE_THRESHOLD} parts):")
    if specialized_machines:
        for m in sorted(specialized_machines, key=lambda m: machine_part_count.get(m, 99)):
            mparts = spec_machine_parts.get(m, [])
            print(f"    {m:<25} Part_Count={machine_part_count.get(m,'?'):<4} parts: {', '.join(mparts)}")
    else:
        print(f"    None")

    # Zero-inv displacement pre-pass
    zero_inv_rr = [
        p for p in active_parts
        if current_inventory.get(p, 0) == 0
        and p not in already_planned
        and part_category.get(p, "Stranger") in ("Runner", "Repeater")
        and vt_compat.get(p)
    ]
    if zero_inv_rr:
        print(f"\n  V11 DISPLACEMENT PRE-PASS  ({len(zero_inv_rr)} zero-inv R/R)")
    else:
        print(f"\n  V11 DISPLACEMENT PRE-PASS  — none  ✓")

    # PRIMARY SCHEDULING PASS
    sorted_active = sorted(
        [p for p in active_parts if p not in already_planned],
        key=lambda p: priority_scores.get(p, 0),
        reverse=True,
    )

    print(f"\n  PRIMARY SCHEDULING PASS  ({len(sorted_active)} active parts)")
    print(f"  {'Part':<30} {'Color':<10} {'Fixed':<20} {'Score':>6} {'Days':>5} "
          f"{'Status':<15} {'Machine(s)':<25} {'Run':>5} {'Qty':>8}")
    print(f"  {'─'*125}")

    for part in sorted_active:
        inv_now  = current_inventory.get(part, 0)
        daily    = indent_daily.get(part, 0)
        monthly  = indent_monthly.get(part, 0)
        score    = priority_scores.get(part, 0)
        tools    = tools_available.get(part, 1)
        category = part_category.get(part, "Stranger")
        color    = part_color.get(part, "UNKNOWN")
        fixed_m  = part_fixed_machine.get(part, "—")
        days_cov = inv_now / daily if daily > 0 else 999

        buf_label = (
            "CRITICAL"     if inv_now == 0 else
            "BELOW_SAFETY" if days_cov < SAFETY_DAYS else
            "BUILDING"     if days_cov < TARGET_DAYS else
            "AT_TARGET"
        )

        if monthly == 0:
            deferred.append({
                "Part": part, "Color": color, "Fixed_Machine": fixed_m,
                "Category": category, "Reason": "Monthly indent = 0",
            })
            print(f"  {part:<30} {color:<10} {fixed_m:<20} {score:>6.1f} {days_cov:>5.1f} "
                  f"{'DEFERRED'